In [41]:
import pandas as pd
import numpy as np
from sklearn.manifold import TSNE

df = pd.read_csv('datasets/cosmetics.csv')
display(df.sample(5))
print(df['Label'].value_counts())

,Label,Brand,Name,Price,Rank,Ingredients,Combination,Dry,Normal,Oily,Sensitive
1248,Eye cream,SHISEIDO,Benefiance NutriPerfect Eye Serum,76,4.0,Visit the Shiseido boutique,0,0,0,0,0
1342,Sun protect,SHISEIDO,Sports BB Broad Spectrum SPF 50+ WetForce,38,4.1,"Water, Dimethicone, Trimethylsiloxysilicate, I...",1,1,1,1,0
804,Treatment,BELIF,Hydra Sebum Control Essence,42,4.3,"Water, Citrus Unshiu Peel Extract, Dipropylene...",0,0,0,0,0
993,Face Mask,OLEHENRIKSEN,Hygge HydraClay™ Detox Mask,32,4.2,"Water, Kaolin, Prunus Amygdalus Dulcis (Sweet ...",1,1,1,1,1
637,Treatment,FRESH,Black Tea Age-Delay Firming Serum,75,3.9,"Water, Glycerin, Butylene Glycol, Jojoba Oil P...",1,1,1,1,1


Label
Moisturizer    298
Cleanser       281
Face Mask      266
Treatment      248
Eye cream      209
Sun protect    170
Name: count, dtype: int64


In [42]:
moisturizers = df[df['Label'] == 'Moisturizer']
moisturizers_dry = moisturizers[moisturizers['Dry'] == 1]
moisturizers_dry = moisturizers_dry.reset_index(drop=True)
print(moisturizers_dry.shape)

(190, 11)


In [43]:
corpus = []
ingredient_idx = {}
idx = 0

for i, row in moisturizers_dry.iterrows():
    ingredients = row['Ingredients'].lower()
    tokens = ingredients.split(', ')
    corpus.append(tokens)
    
    for ingredient in tokens:
        if ingredient not in ingredient_idx:
            ingredient_idx[ingredient] = idx
            idx += 1

print(f"Number of products: {len(corpus)}")
print(f"Number of unique ingredients: {len(ingredient_idx)}")

Number of products: 190
Number of unique ingredients: 2233


In [44]:
M = len(moisturizers_dry)
N = len(ingredient_idx)
A = np.zeros((M, N))
print(f"Matrix shape: {A.shape}")

Matrix shape: (190, 2233)


In [45]:
def oh_encoder(tokens):
    x = np.zeros(N)
    for ingredient in tokens:
        if ingredient in ingredient_idx:
            x[ingredient_idx[ingredient]] = 1
    return x

In [46]:
i = 0
for tokens in corpus:
    A[i] = oh_encoder(tokens)
    i += 1

print("Matrix filled! Shape:", A.shape)

Matrix filled! Shape: (190, 2233)


In [47]:
model = TSNE(n_components=2, learning_rate=200, random_state=42)
tsne_features = model.fit_transform(A)

moisturizers_dry['X'] = tsne_features[:, 0]
moisturizers_dry['Y'] = tsne_features[:, 1]
print("t-SNE done!")

t-SNE done!


In [48]:
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.io import output_notebook

output_notebook()

source = ColumnDataSource(moisturizers_dry)

plot = figure(title="Cosmetic Ingredient Similarity Map",
              x_axis_label='T-SNE 1',
              y_axis_label='T-SNE 2',
              width=700, height=500)

plot.circle(x='X', y='Y', source=source, size=8, alpha=0.6)

Loading BokehJS ...

GlyphRenderer(id='p1237', ...)

In [49]:
hover = HoverTool(tooltips=[
    ('Item', '@Name'),
    ('Brand', '@Brand'),
    ('Price', '$@Price'),
    ('Rank', '@Rank')
])

plot.add_tools(hover)

In [50]:
show(plot)

In [51]:
product1 = "Color Control Cushion Compact Broad Spectrum SPF 50+"
product2 = "BB Cushion Hydra Radiance SPF 50"

for name in [product1, product2]:
    match = moisturizers_dry[moisturizers_dry['Name'] == name]
    if not match.empty:
        print(f"\n--- {name} ---")
        print(match[['Brand', 'Price', 'Rank', 'Ingredients']].to_string())
    else:
        print(f"\n'{name}' not found in moisturizers for dry skin.")


--- Color Control Cushion Compact Broad Spectrum SPF 50+ ---
           Brand  Price  Rank                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [52]:
from bokeh.io import output_file, save

output_file("cosmetic_map.html", title="Cosmetic Ingredient Similarity Map")
save(plot)
print("Chart saved as cosmetic_map.html!")

Chart saved as cosmetic_map.html!


In [53]:
from bokeh.plotting import figure, output_file, save
from bokeh.models import ColumnDataSource, HoverTool, CategoricalColorMapper
from bokeh.palettes import Category20
from bokeh.io import reset_output

# Reset everything
reset_output()

# Get unique brands and assign colors
brands = moisturizers_dry['Brand'].unique().tolist()
palette = Category20[20] * (len(brands) // 20 + 1)
color_mapper = CategoricalColorMapper(factors=brands, palette=palette[:len(brands)])

source = ColumnDataSource(moisturizers_dry)

new_plot = figure(
    title="Cosmetic Ingredient Similarity Map",
    x_axis_label='T-SNE 1',
    y_axis_label='T-SNE 2',
    width=900,
    height=650
)

new_plot.scatter(
    x='X', y='Y',
    source=source,
    size=12,
    alpha=0.8,
    color=dict(field='Brand', transform=color_mapper),
    legend_field='Brand',
    line_color='white',
    line_width=0.5
)

hover = HoverTool(tooltips=[
    ('Product', '@Name'),
    ('Brand', '@Brand'),
    ('Price', '$@Price'),
    ('Rank', '@Rank')
])
new_plot.add_tools(hover)

new_plot.legend.location = "top_left"
new_plot.legend.click_policy = "hide"
new_plot.legend.label_text_font_size = "9px"

output_file("cosmetic_map.html")
save(new_plot)
print("✅ Chart saved! Open cosmetic_map.html in your browser!")

✅ Chart saved! Open cosmetic_map.html in your browser!


In [54]:
import urllib.request
print("We'll create the dashboard directly in Python instead!")

We'll create the dashboard directly in Python instead!


In [55]:
import json

# Get real data
data = moisturizers_dry[['Name','Brand','Price','Rank','Ingredients',
                          'Dry','Normal','Oily','Combination','Sensitive',
                          'X','Y']].to_dict('records')

data_json = json.dumps(data)

# Write dashboard directly
dashboard_html = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Cosmetic Intelligence Dashboard</title>
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Syne:wght@400;600;800&display=swap" rel="stylesheet">
<style>
:root{{--bg:#080b12;--bg2:#0e1220;--bg3:#141824;--accent:#c8f542;--accent2:#42f5c8;--accent3:#f542a0;--text:#e8eaf0;--muted:#6b7280;--border:rgba(200,245,66,0.15);--glow:rgba(200,245,66,0.3);}}
*{{margin:0;padding:0;box-sizing:border-box;}}
body{{background:var(--bg);color:var(--text);font-family:'Syne',sans-serif;min-height:100vh;overflow-x:hidden;}}
body::before{{content:'';position:fixed;inset:0;background-image:linear-gradient(rgba(200,245,66,0.03) 1px,transparent 1px),linear-gradient(90deg,rgba(200,245,66,0.03) 1px,transparent 1px);background-size:40px 40px;pointer-events:none;z-index:0;}}
header{{position:relative;z-index:10;padding:20px 32px;border-bottom:1px solid var(--border);display:flex;align-items:center;justify-content:space-between;background:rgba(8,11,18,0.95);backdrop-filter:blur(12px);}}
.logo{{display:flex;align-items:center;gap:12px;}}
.logo-icon{{width:36px;height:36px;background:var(--accent);border-radius:8px;display:flex;align-items:center;justify-content:center;font-size:18px;}}
.logo-text{{font-size:18px;font-weight:800;color:var(--text);}}
.logo-text span{{color:var(--accent);}}
.header-stats{{display:flex;gap:32px;}}
.hstat{{text-align:center;}}
.hstat-num{{font-family:'Space Mono',monospace;font-size:20px;font-weight:700;color:var(--accent);}}
.hstat-label{{font-size:10px;color:var(--muted);letter-spacing:1px;text-transform:uppercase;}}
.layout{{position:relative;z-index:1;display:grid;grid-template-columns:260px 1fr 280px;height:calc(100vh - 69px);}}
.left-panel{{background:var(--bg2);border-right:1px solid var(--border);overflow-y:auto;padding:20px;}}
.panel-title{{font-size:10px;letter-spacing:2px;text-transform:uppercase;color:var(--accent);margin-bottom:16px;font-family:'Space Mono',monospace;}}
.search-wrap{{position:relative;margin-bottom:20px;}}
.search-wrap input{{width:100%;background:var(--bg3);border:1px solid var(--border);border-radius:8px;padding:10px 14px 10px 36px;color:var(--text);font-family:'Syne',sans-serif;font-size:13px;outline:none;transition:border-color 0.2s;}}
.search-wrap input:focus{{border-color:var(--accent);box-shadow:0 0 0 3px var(--glow);}}
.search-wrap input::placeholder{{color:var(--muted);}}
.search-icon{{position:absolute;left:12px;top:50%;transform:translateY(-50%);color:var(--muted);font-size:14px;}}
.filter-section{{margin-bottom:20px;}}
.filter-label{{font-size:10px;letter-spacing:1.5px;text-transform:uppercase;color:var(--muted);margin-bottom:8px;}}
.filter-chips{{display:flex;flex-wrap:wrap;gap:5px;}}
.chip{{padding:4px 10px;border-radius:20px;font-size:11px;border:1px solid var(--border);background:transparent;color:var(--muted);cursor:pointer;transition:all 0.2s;font-family:'Syne',sans-serif;}}
.chip:hover,.chip.active{{background:var(--accent);border-color:var(--accent);color:#080b12;font-weight:600;}}
.price-range{{display:flex;flex-direction:column;gap:6px;}}
.range-slider{{-webkit-appearance:none;width:100%;height:3px;background:linear-gradient(to right,var(--accent) 0%,var(--accent) 100%,var(--bg3) 100%);border-radius:2px;outline:none;cursor:pointer;}}
.range-slider::-webkit-slider-thumb{{-webkit-appearance:none;width:14px;height:14px;border-radius:50%;background:var(--accent);box-shadow:0 0 8px var(--glow);cursor:pointer;}}
.price-labels{{display:flex;justify-content:space-between;font-size:10px;color:var(--muted);font-family:'Space Mono',monospace;}}
.brand-list{{display:flex;flex-direction:column;gap:3px;max-height:180px;overflow-y:auto;}}
.brand-item{{display:flex;align-items:center;gap:8px;padding:5px 6px;border-radius:6px;cursor:pointer;transition:background 0.15s;font-size:11px;}}
.brand-item:hover{{background:var(--bg3);}}
.brand-dot{{width:8px;height:8px;border-radius:50%;flex-shrink:0;}}
.brand-name{{color:var(--text);flex:1;}}
.brand-count{{font-family:'Space Mono',monospace;font-size:10px;color:var(--muted);}}
.brand-item.selected .brand-name{{color:var(--accent);}}
.center-panel{{display:flex;flex-direction:column;overflow:hidden;}}
.viz-toolbar{{padding:10px 16px;border-bottom:1px solid var(--border);display:flex;align-items:center;gap:10px;background:var(--bg2);}}
.viz-label{{font-size:11px;color:var(--muted);margin-right:auto;font-family:'Space Mono',monospace;}}
.viz-btn{{padding:5px 12px;border-radius:6px;font-size:11px;border:1px solid var(--border);background:transparent;color:var(--muted);cursor:pointer;font-family:'Syne',sans-serif;transition:all 0.2s;}}
.viz-btn:hover,.viz-btn.active{{border-color:var(--accent);color:var(--accent);}}
canvas#scatterCanvas{{flex:1;display:block;cursor:crosshair;}}
.right-panel{{background:var(--bg2);border-left:1px solid var(--border);overflow-y:auto;display:flex;flex-direction:column;}}
.stats-row{{display:grid;grid-template-columns:1fr 1fr;gap:1px;border-bottom:1px solid var(--border);}}
.stat-card{{padding:14px;background:var(--bg2);}}
.stat-num{{font-family:'Space Mono',monospace;font-size:20px;font-weight:700;color:var(--accent);}}
.stat-num.teal{{color:var(--accent2);}}
.stat-num.pink{{color:var(--accent3);}}
.stat-desc{{font-size:10px;color:var(--muted);margin-top:2px;}}
.detail-area{{flex:1;padding:16px;}}
.no-selection{{height:200px;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:10px;color:var(--muted);text-align:center;}}
.no-selection-icon{{font-size:36px;opacity:0.3;}}
.no-selection-text{{font-size:12px;line-height:1.6;}}
.product-card{{display:none;}}
.product-card.visible{{display:block;}}
.product-brand{{font-size:10px;letter-spacing:2px;text-transform:uppercase;color:var(--accent);font-family:'Space Mono',monospace;margin-bottom:4px;}}
.product-name{{font-size:15px;font-weight:800;line-height:1.3;color:var(--text);}}
.product-meta{{display:flex;gap:8px;margin-top:10px;margin-bottom:16px;}}
.meta-pill{{padding:3px 9px;border-radius:20px;font-size:11px;font-family:'Space Mono',monospace;}}
.meta-pill.price{{background:rgba(200,245,66,0.1);color:var(--accent);border:1px solid rgba(200,245,66,0.2);}}
.meta-pill.rank{{background:rgba(66,245,200,0.1);color:var(--accent2);border:1px solid rgba(66,245,200,0.2);}}
.section-title{{font-size:10px;letter-spacing:2px;text-transform:uppercase;color:var(--muted);margin-bottom:8px;}}
.ingredient-tags{{display:flex;flex-wrap:wrap;gap:4px;margin-bottom:16px;}}
.ing-tag{{padding:3px 7px;background:var(--bg3);border:1px solid var(--border);border-radius:4px;font-size:10px;color:var(--muted);font-family:'Space Mono',monospace;}}
.ing-tag.highlight{{background:rgba(200,245,66,0.08);border-color:rgba(200,245,66,0.3);color:var(--accent);}}
.similar-list{{display:flex;flex-direction:column;gap:5px;}}
.similar-item{{padding:8px;background:var(--bg3);border:1px solid var(--border);border-radius:8px;cursor:pointer;transition:all 0.2s;}}
.similar-item:hover{{border-color:var(--accent);background:rgba(200,245,66,0.04);}}
.similar-item-brand{{font-size:9px;color:var(--accent);letter-spacing:1px;text-transform:uppercase;}}
.similar-item-name{{font-size:11px;color:var(--text);margin-top:2px;}}
.similar-item-price{{font-size:10px;color:var(--muted);font-family:'Space Mono',monospace;margin-top:3px;}}
#tooltip{{position:fixed;background:var(--bg2);border:1px solid var(--accent);border-radius:8px;padding:8px 12px;pointer-events:none;z-index:1000;display:none;max-width:200px;box-shadow:0 0 20px var(--glow);}}
.tt-brand{{font-size:9px;color:var(--accent);letter-spacing:1.5px;text-transform:uppercase;font-family:'Space Mono',monospace;}}
.tt-name{{font-size:12px;color:var(--text);margin-top:2px;font-weight:600;}}
.tt-price{{font-size:11px;color:var(--accent2);margin-top:3px;font-family:'Space Mono',monospace;}}
::-webkit-scrollbar{{width:3px;}}
::-webkit-scrollbar-track{{background:var(--bg);}}
::-webkit-scrollbar-thumb{{background:var(--border);border-radius:2px;}}
.bottom-bar{{position:fixed;bottom:0;left:0;right:0;height:28px;background:var(--bg2);border-top:1px solid var(--border);display:flex;align-items:center;padding:0 20px;gap:20px;z-index:100;font-family:'Space Mono',monospace;font-size:10px;color:var(--muted);}}
.bottom-bar span{{color:var(--accent);}}
.status-dot{{width:6px;height:6px;border-radius:50%;background:var(--accent);box-shadow:0 0 6px var(--accent);animation:pulse 2s infinite;}}
@keyframes pulse{{0%,100%{{opacity:1;}}50%{{opacity:0.4;}}}}
</style>
</head>
<body>
<header>
  <div class="logo">
    <div class="logo-icon">🧬</div>
    <div class="logo-text">Cosmetic<span>Lab</span></div>
  </div>
  <div class="header-stats">
    <div class="hstat"><div class="hstat-num" id="totalCount">190</div><div class="hstat-label">Products</div></div>
    <div class="hstat"><div class="hstat-num" id="brandCount">—</div><div class="hstat-label">Brands</div></div>
    <div class="hstat"><div class="hstat-num" id="visibleCount">190</div><div class="hstat-label">Visible</div></div>
    <div class="hstat"><div class="hstat-num" id="avgPrice">—</div><div class="hstat-label">Avg Price</div></div>
  </div>
</header>
<div class="layout">
  <div class="left-panel">
    <div class="panel-title">// Filters</div>
    <div class="search-wrap">
      <span class="search-icon">🔍</span>
      <input type="text" id="searchInput" placeholder="Search products or brands..." oninput="applyFilters()">
    </div>
    <div class="filter-section">
      <div class="filter-label">Skin Type</div>
      <div class="filter-chips">
        <div class="chip active" onclick="toggleSkin(this,'all')">All</div>
        <div class="chip" onclick="toggleSkin(this,'Dry')">Dry</div>
        <div class="chip" onclick="toggleSkin(this,'Normal')">Normal</div>
        <div class="chip" onclick="toggleSkin(this,'Oily')">Oily</div>
        <div class="chip" onclick="toggleSkin(this,'Combination')">Combo</div>
        <div class="chip" onclick="toggleSkin(this,'Sensitive')">Sensitive</div>
      </div>
    </div>
    <div class="filter-section">
      <div class="filter-label">Price Range</div>
      <div class="price-range">
        <input type="range" class="range-slider" id="priceRange" min="0" max="500" value="500" oninput="updatePrice(this)">
        <div class="price-labels"><span>$0</span><span id="priceLabel">Any price</span></div>
      </div>
    </div>
    <div class="filter-section">
      <div class="filter-label">Color By</div>
      <div class="filter-chips">
        <div class="chip active" onclick="setColorMode(this,'brand')">Brand</div>
        <div class="chip" onclick="setColorMode(this,'price')">Price</div>
        <div class="chip" onclick="setColorMode(this,'rank')">Rank</div>
      </div>
    </div>
    <div class="filter-section">
      <div class="filter-label">Brands</div>
      <div class="brand-list" id="brandList"></div>
    </div>
  </div>
  <div class="center-panel">
    <div class="viz-toolbar">
      <span class="viz-label">t-SNE · Ingredient Similarity Map</span>
      <button class="viz-btn active" onclick="setVizMode(this,'scatter')">Scatter</button>
      <button class="viz-btn" onclick="setVizMode(this,'density')">Density</button>
      <button class="viz-btn" onclick="resetZoom()">Reset View</button>
    </div>
    <canvas id="scatterCanvas"></canvas>
  </div>
  <div class="right-panel">
    <div class="stats-row">
      <div class="stat-card"><div class="stat-num" id="statAvgRank">—</div><div class="stat-desc">Avg Rank</div></div>
      <div class="stat-card"><div class="stat-num teal" id="statMinPrice">—</div><div class="stat-desc">Min Price</div></div>
      <div class="stat-card"><div class="stat-num pink" id="statMaxPrice">—</div><div class="stat-desc">Max Price</div></div>
      <div class="stat-card"><div class="stat-num" id="statIngCount">—</div><div class="stat-desc">Avg Ingredients</div></div>
    </div>
    <div class="detail-area">
      <div class="no-selection" id="noSelection">
        <div class="no-selection-icon">✦</div>
        <div class="no-selection-text">Click any dot on the map<br>to explore product details</div>
      </div>
      <div class="product-card" id="productCard">
        <div class="product-brand" id="pdBrand"></div>
        <div class="product-name" id="pdName"></div>
        <div class="product-meta">
          <div class="meta-pill price" id="pdPrice"></div>
          <div class="meta-pill rank" id="pdRank"></div>
        </div>
        <div class="section-title">// Ingredients</div>
        <div class="ingredient-tags" id="pdIngredients"></div>
        <div class="section-title">// Similar Products</div>
        <div class="similar-list" id="similarList"></div>
      </div>
    </div>
  </div>
</div>
<div id="tooltip">
  <div class="tt-brand" id="ttBrand"></div>
  <div class="tt-name" id="ttName"></div>
  <div class="tt-price" id="ttPrice"></div>
</div>
<div class="bottom-bar">
  <div class="status-dot"></div>
  <span>COSMETIC<span>LAB</span></span>
  <span>Sephora · 190 Moisturizers · Dry Skin</span>
  <span style="margin-left:auto">Visible: <span id="footerVisible">190</span></span>
</div>
<script>
const PRODUCTS = {data_json};
const BRAND_COLORS = ['#c8f542','#42f5c8','#f542a0','#f5a742','#4287f5','#f54242','#42f587','#a742f5','#f5e642','#42c8f5','#f5426b','#6bf542','#f5a042','#4242f5','#f542e0','#42f5a0','#f56b42','#42a0f5','#c842f5','#f5c842','#42f542','#f54287','#87f542'];
let state={{search:'',skinFilter:'all',maxPrice:500,colorMode:'brand',selectedBrands:new Set(),vizMode:'scatter',selectedIdx:-1,zoom:1,panX:0,panY:0,dragging:false,dragStart:null}};
let filteredIndices=[],brandColorMap={{}},canvas,ctx,hoveredIdx=-1;
window.addEventListener('load',()=>{{
  canvas=document.getElementById('scatterCanvas');
  ctx=canvas.getContext('2d');
  resizeCanvas();
  window.addEventListener('resize',resizeCanvas);
  canvas.addEventListener('mousemove',onMouseMove);
  canvas.addEventListener('click',onCanvasClick);
  canvas.addEventListener('wheel',onWheel,{{passive:false}});
  canvas.addEventListener('mousedown',e=>{{state.dragging=true;state.dragStart={{x:e.clientX-state.panX,y:e.clientY-state.panY}};}});
  canvas.addEventListener('mouseup',()=>state.dragging=false);
  buildBrandList();updateHeaderStats();applyFilters();
}});
function buildBrandList(){{
  const counts={{}};
  PRODUCTS.forEach(p=>{{counts[p.Brand]=(counts[p.Brand]||0)+1;}});
  const bl=Object.entries(counts).sort((a,b)=>b[1]-a[1]);
  bl.forEach(([b],i)=>{{brandColorMap[b]=BRAND_COLORS[i%BRAND_COLORS.length];}});
  document.getElementById('brandList').innerHTML=bl.map(([b,c],i)=>`<div class="brand-item" onclick="toggleBrand('${{b.replace(/'/g,"\\'")}}',this)"><div class="brand-dot" style="background:${{BRAND_COLORS[i%BRAND_COLORS.length]}}"></div><span class="brand-name">${{b}}</span><span class="brand-count">${{c}}</span></div>`).join('');
  document.getElementById('brandCount').textContent=bl.length;
}}
function updateHeaderStats(){{
  const prices=PRODUCTS.map(p=>p.Price);
  document.getElementById('avgPrice').textContent='$'+Math.round(prices.reduce((a,b)=>a+b,0)/prices.length);
  const ranks=PRODUCTS.map(p=>p.Rank);
  document.getElementById('statAvgRank').textContent=(ranks.reduce((a,b)=>a+b,0)/ranks.length).toFixed(1)+'★';
  document.getElementById('statMinPrice').textContent='$'+Math.min(...prices);
  document.getElementById('statMaxPrice').textContent='$'+Math.max(...prices);
  const ings=PRODUCTS.map(p=>p.Ingredients.split(',').length);
  document.getElementById('statIngCount').textContent=Math.round(ings.reduce((a,b)=>a+b,0)/ings.length);
}}
function applyFilters(){{
  state.search=document.getElementById('searchInput').value.toLowerCase();
  filteredIndices=PRODUCTS.map((_,i)=>i).filter(i=>{{
    const p=PRODUCTS[i];
    if(state.search&&!p.Name.toLowerCase().includes(state.search)&&!p.Brand.toLowerCase().includes(state.search))return false;
    if(state.skinFilter!=='all'&&!p[state.skinFilter])return false;
    if(p.Price>state.maxPrice)return false;
    if(state.selectedBrands.size>0&&!state.selectedBrands.has(p.Brand))return false;
    return true;
  }});
  document.getElementById('visibleCount').textContent=filteredIndices.length;
  document.getElementById('footerVisible').textContent=filteredIndices.length;
  drawCanvas();
}}
function toggleSkin(el,val){{document.querySelectorAll('.filter-chips .chip').forEach(c=>{{if(c.onclick&&c.onclick.toString().includes('toggleSkin'))c.classList.remove('active');}});el.classList.add('active');state.skinFilter=val;applyFilters();}}
function updatePrice(el){{state.maxPrice=parseInt(el.value);const pct=state.maxPrice/500*100;el.style.background=`linear-gradient(to right,var(--accent) 0%,var(--accent) ${{pct}}%,var(--bg3) ${{pct}}%)`;document.getElementById('priceLabel').textContent=state.maxPrice>=500?'Any price':`Up to $${{state.maxPrice}}`;applyFilters();}}
function toggleBrand(brand,el){{el.classList.toggle('selected');if(state.selectedBrands.has(brand))state.selectedBrands.delete(brand);else state.selectedBrands.add(brand);applyFilters();}}
function setColorMode(el,mode){{document.querySelectorAll('.filter-chips .chip').forEach(c=>{{if(c.onclick&&c.onclick.toString().includes('setColorMode'))c.classList.remove('active');}});el.classList.add('active');state.colorMode=mode;drawCanvas();}}
function setVizMode(el,mode){{document.querySelectorAll('.viz-btn').forEach(b=>b.classList.remove('active'));el.classList.add('active');state.vizMode=mode;drawCanvas();}}
function resetZoom(){{state.zoom=1;state.panX=0;state.panY=0;drawCanvas();}}
function resizeCanvas(){{const p=canvas.parentElement;canvas.width=p.clientWidth;canvas.height=p.clientHeight-41;drawCanvas();}}
function getColor(p){{
  if(state.colorMode==='brand')return brandColorMap[p.Brand]||'#c8f542';
  if(state.colorMode==='price'){{const t=Math.min(p.Price/350,1);return `rgb(${{Math.round(66+(245-66)*t)}},${{Math.round(245+(66-245)*t)}},200)`;}}
  if(state.colorMode==='rank'){{const t=(p.Rank-3)/2;return `rgb(${{Math.round(245+(200-245)*t)}},${{Math.round(66+(245-66)*t)}},66)`;}}
  return '#c8f542';
}}
function worldToScreen(x,y){{const cx=canvas.width/2+state.panX,cy=canvas.height/2+state.panY,sc=(Math.min(canvas.width,canvas.height)/900)*state.zoom;return{{x:cx+x*sc,y:cy+y*sc}};}}
function drawCanvas(){{
  if(!ctx)return;
  ctx.clearRect(0,0,canvas.width,canvas.height);
  const fs=new Set(filteredIndices);
  PRODUCTS.forEach((p,i)=>{{
    if(fs.has(i))return;
    const {{x,y}}=worldToScreen(p.X,p.Y);
    ctx.beginPath();ctx.arc(x,y,3,0,Math.PI*2);ctx.fillStyle='rgba(100,110,130,0.12)';ctx.fill();
  }});
  if(state.vizMode==='density'){{
    filteredIndices.forEach(i=>{{
      const p=PRODUCTS[i],{{x,y}}=worldToScreen(p.X,p.Y),color=getColor(p);
      const g=ctx.createRadialGradient(x,y,0,x,y,28);
      g.addColorStop(0,color+'35');g.addColorStop(1,color+'00');
      ctx.beginPath();ctx.arc(x,y,28,0,Math.PI*2);ctx.fillStyle=g;ctx.fill();
    }});
  }}
  filteredIndices.forEach(i=>{{
    const p=PRODUCTS[i],{{x,y}}=worldToScreen(p.X,p.Y),color=getColor(p),isSel=state.selectedIdx===i,r=isSel?10:6;
    if(isSel){{const g=ctx.createRadialGradient(x,y,0,x,y,22);g.addColorStop(0,color+'70');g.addColorStop(1,color+'00');ctx.beginPath();ctx.arc(x,y,22,0,Math.PI*2);ctx.fillStyle=g;ctx.fill();}}
    ctx.beginPath();ctx.arc(x,y,r,0,Math.PI*2);ctx.fillStyle=isSel?color:color+'bb';ctx.fill();
    if(isSel){{ctx.beginPath();ctx.arc(x,y,r+3,0,Math.PI*2);ctx.strokeStyle=color;ctx.lineWidth=1.5;ctx.stroke();}}
  }});
}}
function onMouseMove(e){{
  if(state.dragging&&state.dragStart){{state.panX=e.clientX-state.dragStart.x;state.panY=e.clientY-state.dragStart.y;drawCanvas();return;}}
  const rect=canvas.getBoundingClientRect(),mx=e.clientX-rect.left,my=e.clientY-rect.top;
  let closest=-1,cd=18;
  filteredIndices.forEach(i=>{{const p=PRODUCTS[i],{{x,y}}=worldToScreen(p.X,p.Y),d=Math.hypot(mx-x,my-y);if(d<cd){{closest=i;cd=d;}}}});
  const tt=document.getElementById('tooltip');
  if(closest>=0){{
    canvas.style.cursor='pointer';
    const p=PRODUCTS[closest];
    document.getElementById('ttBrand').textContent=p.Brand;
    document.getElementById('ttName').textContent=p.Name;
    document.getElementById('ttPrice').textContent=`$${{p.Price}} · ★${{p.Rank}}`;
    tt.style.display='block';tt.style.left=(e.clientX+14)+'px';tt.style.top=(e.clientY-10)+'px';
    hoveredIdx=closest;
  }}else{{canvas.style.cursor='crosshair';tt.style.display='none';hoveredIdx=-1;}}
}}
function onCanvasClick(){{if(hoveredIdx>=0){{state.selectedIdx=hoveredIdx;showDetail(PRODUCTS[hoveredIdx],hoveredIdx);drawCanvas();}}}}
function onWheel(e){{e.preventDefault();state.zoom=Math.max(0.3,Math.min(5,state.zoom*(e.deltaY>0?0.9:1.1)));drawCanvas();}}
function showDetail(p,idx){{
  document.getElementById('noSelection').style.display='none';
  const card=document.getElementById('productCard');card.classList.add('visible');
  document.getElementById('pdBrand').textContent=p.Brand;
  document.getElementById('pdName').textContent=p.Name;
  document.getElementById('pdPrice').textContent='$'+p.Price;
  document.getElementById('pdRank').textContent='★'+p.Rank;
  const ings=p.Ingredients.split(',').map(s=>s.trim()).filter(Boolean);
  const keys=['Water','Glycerin','Niacinamide','Hyaluronic Acid','Ceramide','Retinol','Vitamin C','Peptides','Squalane'];
  document.getElementById('pdIngredients').innerHTML=ings.slice(0,18).map(ing=>`<span class="ing-tag ${{keys.some(k=>ing.toLowerCase().includes(k.toLowerCase()))?'highlight':''}}">${{ing}}</span>`).join('');
  const dists=PRODUCTS.map((q,i)=>{{return{{i,d:Math.hypot(q.X-p.X,q.Y-p.Y)}}}}).filter(d=>d.i!==idx).sort((a,b)=>a.d-b.d).slice(0,4);
  document.getElementById('similarList').innerHTML=dists.map(({{i}})=>{{const q=PRODUCTS[i];return `<div class="similar-item" onclick="selectProduct(${{i}})"><div class="similar-item-brand">${{q.Brand}}</div><div class="similar-item-name">${{q.Name}}</div><div class="similar-item-price">$${{q.Price}} · ★${{q.Rank}}</div></div>`;}}).join('');
}}
function selectProduct(idx){{state.selectedIdx=idx;showDetail(PRODUCTS[idx],idx);drawCanvas();}}
</script>
</body>
</html>'''

with open('cosmetic_dashboard.html', 'w') as f:
    f.write(dashboard_html)

print("✅ Dashboard created! Open cosmetic_dashboard.html in your browser!")

✅ Dashboard created! Open cosmetic_dashboard.html in your browser!


In [56]:
import json

data = moisturizers_dry[['Name','Brand','Price','Rank','Ingredients',
                          'Dry','Normal','Oily','Combination','Sensitive',
                          'X','Y']].to_dict('records')

# Fix types
for p in data:
    p['Price'] = float(p['Price'])
    p['Rank'] = float(p['Rank'])
    p['Dry'] = int(p['Dry'])
    p['Normal'] = int(p['Normal'])
    p['Oily'] = int(p['Oily'])
    p['Combination'] = int(p['Combination'])
    p['Sensitive'] = int(p['Sensitive'])

data_json = json.dumps(data)

# Read dashboard
with open('cosmetic_dashboard.html', 'r') as f:
    html = f.read()

# Find and replace safely
start = html.find('const PRODUCTS = ')
end = html.find(';', start) + 1
new_html = html[:start] + f'const PRODUCTS = {data_json}' + html[end-1:]

with open('cosmetic_dashboard.html', 'w') as f:
    f.write(new_html)

print("✅ Fixed! Refresh your browser!")

✅ Fixed! Refresh your browser!


In [57]:
# Read dashboard
with open('cosmetic_dashboard.html', 'r') as f:
    html = f.read()

# Fix the color function to use actual min/max of YOUR data
min_price = float(moisturizers_dry['Price'].min())
max_price = float(moisturizers_dry['Price'].max())
min_rank = float(moisturizers_dry['Rank'].min())
max_rank = float(moisturizers_dry['Rank'].max())

old_color = "if(state.colorMode==='price'){const t=Math.min(p.Price/350,1);return `rgb(${Math.round(66+(245-66)*t)},${Math.round(245+(66-245)*t)},200)`;}"

new_color = f"if(state.colorMode==='price'){{const t=(p.Price-{min_price})/({max_price}-{min_price});const r=Math.round(66+(245-66)*t);const g=Math.round(245+(66-245)*t);return `rgb(${{r}},${{g}},80)`;}}if(state.colorMode==='rank'){{const t=(p.Rank-{min_rank})/({max_rank}-{min_rank});return `rgb(${{Math.round(245*t)}},${{Math.round(200*(1-t)+66*t)}},255)`;}}"

html = html.replace(old_color, new_color)

with open('cosmetic_dashboard.html', 'w') as f:
    f.write(html)

print(f"✅ Fixed! Price range: ${min_price:.0f} - ${max_price:.0f}")
print("Refresh your browser!")

print("Price Distribution:")
print(moisturizers_dry['Price'].describe())
print("\nHow many products over $100:", (moisturizers_dry['Price'] > 100).sum())
print("How many products under $50:", (moisturizers_dry['Price'] < 50).sum())

✅ Fixed! Price range: $10 - $290
Refresh your browser!
Price Distribution:
count    190.000000
mean      70.157895
std       49.943209
min       10.000000
25%       39.000000
50%       52.000000
75%       85.000000
max      290.000000
Name: Price, dtype: float64

How many products over $100: 29
How many products under $50: 90


In [58]:
import json

# Prepare chart data
brand_counts = moisturizers_dry['Brand'].value_counts().head(10)
brand_data = [{"brand": b, "count": int(c)} for b, c in brand_counts.items()]

# Price buckets
bins = [0, 30, 60, 100, 200, 500]
labels = ['$0-30', '$30-60', '$60-100', '$100-200', '$200+']
price_data = []
for i in range(len(bins)-1):
    count = int(((moisturizers_dry['Price'] >= bins[i]) & (moisturizers_dry['Price'] < bins[i+1])).sum())
    price_data.append({"label": labels[i], "count": count})

# Rank distribution
rank_counts = moisturizers_dry['Rank'].value_counts().sort_index()
rank_data = [{"rank": str(round(float(r),1)), "count": int(c)} for r, c in rank_counts.items()]

# Top ingredients
from collections import Counter
all_ingredients = []
for ing_list in moisturizers_dry['Ingredients']:
    for ing in ing_list.split(','):
        ing = ing.strip().lower()
        if len(ing) > 2:
            all_ingredients.append(ing)
top_ingredients = Counter(all_ingredients).most_common(15)
ing_data = [{"name": k.title(), "count": v} for k, v in top_ingredients]

charts_json = json.dumps({
    "brands": brand_data,
    "prices": price_data,
    "ranks": rank_data,
    "ingredients": ing_data
})

print("Brand data:", brand_data[:3])
print("Price data:", price_data)
print("✅ Chart data ready!")

Brand data: [{'brand': 'ORIGINS', 'count': 9}, {'brand': 'CAUDALIE', 'count': 8}, {'brand': 'SHISEIDO', 'count': 8}]
Price data: [{'label': '$0-30', 'count': 14}, {'label': '$30-60', 'count': 96}, {'label': '$60-100', 'count': 50}, {'label': '$100-200', 'count': 23}, {'label': '$200+', 'count': 7}]
✅ Chart data ready!


In [59]:
# Read existing dashboard
with open('cosmetic_dashboard.html', 'r') as f:
    html = f.read()

charts_section = f'''
<div id="chartsPanel" style="position:relative;z-index:1;background:#0e1220;border-top:1px solid rgba(200,245,66,0.15);padding:24px 32px 40px;">
  <div style="display:flex;align-items:center;gap:12px;margin-bottom:24px;">
    <span style="font-family:'Space Mono',monospace;font-size:10px;letter-spacing:2px;color:#c8f542;">// ANALYTICS OVERVIEW</span>
    <div style="flex:1;height:1px;background:rgba(200,245,66,0.1);"></div>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:20px;">

    <!-- BAR CHART: Top Brands -->
    <div style="background:#141824;border:1px solid rgba(200,245,66,0.12);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#6b7280;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:16px;">Top 10 Brands</div>
      <canvas id="brandChart" height="220"></canvas>
    </div>

    <!-- PIE CHART: Price Range -->
    <div style="background:#141824;border:1px solid rgba(200,245,66,0.12);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#6b7280;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:16px;">Price Distribution</div>
      <canvas id="priceChart" height="220"></canvas>
    </div>

    <!-- BAR CHART: Rating Distribution -->
    <div style="background:#141824;border:1px solid rgba(200,245,66,0.12);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#6b7280;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:16px;">Rating Distribution</div>
      <canvas id="rankChart" height="220"></canvas>
    </div>

    <!-- BAR CHART: Top Ingredients -->
    <div style="background:#141824;border:1px solid rgba(200,245,66,0.12);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#6b7280;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:16px;">Top 15 Ingredients</div>
      <canvas id="ingChart" height="220"></canvas>
    </div>

  </div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<script>
const CHART_DATA = {charts_json};

Chart.defaults.color = '#6b7280';
Chart.defaults.borderColor = 'rgba(200,245,66,0.08)';
Chart.defaults.font.family = "'Space Mono', monospace";
Chart.defaults.font.size = 10;

// Brand bar chart
new Chart(document.getElementById('brandChart'), {{
  type: 'bar',
  data: {{
    labels: CHART_DATA.brands.map(d => d.brand),
    datasets: [{{
      data: CHART_DATA.brands.map(d => d.count),
      backgroundColor: CHART_DATA.brands.map((_,i) => `hsl(${{i*36}},80%,60%)`),
      borderRadius: 4,
      borderSkipped: false,
    }}]
  }},
  options: {{
    indexAxis: 'y',
    responsive: true,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ grid: {{ color: 'rgba(200,245,66,0.06)' }}, ticks: {{ color: '#6b7280' }} }},
      y: {{ grid: {{ display: false }}, ticks: {{ color: '#c8f542', font: {{ size: 9 }} }} }}
    }}
  }}
}});

// Price pie chart
new Chart(document.getElementById('priceChart'), {{
  type: 'doughnut',
  data: {{
    labels: CHART_DATA.prices.map(d => d.label),
    datasets: [{{
      data: CHART_DATA.prices.map(d => d.count),
      backgroundColor: ['#42f5c8','#c8f542','#f5a742','#f542a0','#4287f5'],
      borderColor: '#141824',
      borderWidth: 3,
    }}]
  }},
  options: {{
    responsive: true,
    plugins: {{
      legend: {{
        position: 'bottom',
        labels: {{ color: '#6b7280', font: {{ size: 9 }}, boxWidth: 10, padding: 8 }}
      }}
    }}
  }}
}});

// Rank bar chart
new Chart(document.getElementById('rankChart'), {{
  type: 'bar',
  data: {{
    labels: CHART_DATA.ranks.map(d => d.rank),
    datasets: [{{
      data: CHART_DATA.ranks.map(d => d.count),
      backgroundColor: CHART_DATA.ranks.map(d => {{
        const t = (parseFloat(d.rank) - 3) / 2;
        const r = Math.round(245 * (1-t) + 66 * t);
        const g = Math.round(66 * (1-t) + 245 * t);
        return `rgb(${{r}},${{g}},80)`;
      }}),
      borderRadius: 4,
      borderSkipped: false,
    }}]
  }},
  options: {{
    responsive: true,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ grid: {{ display: false }}, ticks: {{ color: '#c8f542' }} }},
      y: {{ grid: {{ color: 'rgba(200,245,66,0.06)' }}, ticks: {{ color: '#6b7280' }} }}
    }}
  }}
}});

// Ingredients bar chart
new Chart(document.getElementById('ingChart'), {{
  type: 'bar',
  data: {{
    labels: CHART_DATA.ingredients.map(d => d.name),
    datasets: [{{
      data: CHART_DATA.ingredients.map(d => d.count),
      backgroundColor: '#42f5c8aa',
      borderColor: '#42f5c8',
      borderWidth: 1,
      borderRadius: 4,
      borderSkipped: false,
    }}]
  }},
  options: {{
    indexAxis: 'y',
    responsive: true,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ grid: {{ color: 'rgba(200,245,66,0.06)' }}, ticks: {{ color: '#6b7280' }} }},
      y: {{ grid: {{ display: false }}, ticks: {{ color: '#42f5c8', font: {{ size: 9 }} }} }}
    }}
  }}
}});
</script>
'''

# Inject before closing body tag
html = html.replace('</body>', charts_section + '</body>')

with open('cosmetic_dashboard.html', 'w') as f:
    f.write(html)

print("✅ Charts added! Refresh your browser!")

✅ Charts added! Refresh your browser!


In [60]:
from collections import Counter
import json

# Key beneficial ingredients to track
key_ingredients = {
    'Hyaluronic Acid': '💧 Hydration',
    'Glycerin': '💧 Hydration', 
    'Ceramide': '🛡️ Barrier',
    'Niacinamide': '✨ Brightening',
    'Retinol': '⏳ Anti-aging',
    'Vitamin C': '✨ Brightening',
    'Peptides': '⏳ Anti-aging',
    'Squalane': '🛡️ Barrier',
    'Aloe': '🌿 Soothing',
    'Zinc': '🧴 Oil Control',
    'Salicylic Acid': '🧴 Oil Control',
    'Collagen': '⏳ Anti-aging',
    'Vitamin E': '🛡️ Barrier',
    'SPF': '☀️ Sun Protection',
    'Caffeine': '👁️ Depuffing'
}

# Score each product
scores = []
for _, row in moisturizers_dry.iterrows():
    ing_lower = row['Ingredients'].lower()
    found = []
    for ing, benefit in key_ingredients.items():
        if ing.lower() in ing_lower:
            found.append({'ingredient': ing, 'benefit': benefit})
    scores.append({
        'Name': row['Name'],
        'Brand': row['Brand'],
        'Price': float(row['Price']),
        'Rank': float(row['Rank']),
        'score': len(found),
        'key_ingredients': found
    })

# Top scoring products
top_products = sorted(scores, key=lambda x: x['score'], reverse=True)[:10]

# Benefit counts
benefit_counts = Counter()
for s in scores:
    for ki in s['key_ingredients']:
        benefit_counts[ki['benefit']] += 1

# Brand ingredient overlap
brands_top = moisturizers_dry['Brand'].value_counts().head(8).index.tolist()
heatmap_data = []
for b1 in brands_top:
    row_data = []
    ings1 = set()
    for _, r in moisturizers_dry[moisturizers_dry['Brand']==b1].iterrows():
        ings1.update([i.strip().lower() for i in r['Ingredients'].split(',')])
    for b2 in brands_top:
        ings2 = set()
        for _, r in moisturizers_dry[moisturizers_dry['Brand']==b2].iterrows():
            ings2.update([i.strip().lower() for i in r['Ingredients'].split(',')])
        overlap = len(ings1 & ings2) / max(len(ings1 | ings2), 1)
        row_data.append(round(overlap * 100, 1))
    heatmap_data.append(row_data)

level2_json = json.dumps({
    'top_products': top_products,
    'benefit_counts': dict(benefit_counts),
    'brands': brands_top,
    'heatmap': heatmap_data
})

print("Top 3 products by key ingredients:")
for p in top_products[:3]:
    print(f"  {p['Name']} — score: {p['score']}")
print("✅ Level 2 data ready!")

Top 3 products by key ingredients:
  Miracle Water 3-in-1 Micellar Cleanser — score: 7
  The Littles™ — score: 6
  Your Skin But Better CC+ Cream Oil-Free Matte with SPF 40 — score: 6
✅ Level 2 data ready!


In [61]:
# Read dashboard
with open('cosmetic_dashboard.html', 'r') as f:
    html = f.read()

level2_section = f'''
<div style="position:relative;z-index:1;background:#080b12;border-top:1px solid rgba(200,245,66,0.15);padding:24px 32px 40px;">
  <div style="display:flex;align-items:center;gap:12px;margin-bottom:24px;">
    <span style="font-family:'Space Mono',monospace;font-size:10px;letter-spacing:2px;color:#42f5c8;">// INGREDIENT INTELLIGENCE</span>
    <div style="flex:1;height:1px;background:rgba(66,245,200,0.1);"></div>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">

    <!-- TOP SCORED PRODUCTS -->
    <div style="background:#141824;border:1px solid rgba(66,245,200,0.12);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#6b7280;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:16px;">🏆 Top Products by Beneficial Ingredients</div>
      <div id="topProductsList"></div>
    </div>

    <!-- INGREDIENT HEATMAP -->
    <div style="background:#141824;border:1px solid rgba(66,245,200,0.12);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#6b7280;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:16px;">🔥 Brand Ingredient Overlap Heatmap</div>
      <div id="heatmapContainer"></div>
    </div>

    <!-- BENEFIT BREAKDOWN -->
    <div style="background:#141824;border:1px solid rgba(66,245,200,0.12);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#6b7280;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:16px;">💊 Benefit Category Breakdown</div>
      <canvas id="benefitChart" height="200"></canvas>
    </div>

    <!-- KEY INGREDIENT FINDER -->
    <div style="background:#141824;border:1px solid rgba(66,245,200,0.12);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#6b7280;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:12px;">🔍 Find Products by Ingredient</div>
      <input type="text" id="ingSearch" placeholder="Type ingredient e.g. Retinol, Ceramide..." 
        style="width:100%;background:#0e1220;border:1px solid rgba(200,245,66,0.2);border-radius:8px;padding:10px 14px;color:#e8eaf0;font-family:'Space Mono',monospace;font-size:12px;outline:none;margin-bottom:12px;">
      <div id="ingResults" style="display:flex;flex-direction:column;gap:6px;max-height:200px;overflow-y:auto;"></div>
    </div>

  </div>
</div>

<script>
const L2_DATA = {level2_json};

// Top Products List
const topList = document.getElementById('topProductsList');
topList.innerHTML = L2_DATA.top_products.slice(0,8).map((p,i) => `
  <div style="display:flex;align-items:center;gap:10px;padding:8px 0;border-bottom:1px solid rgba(200,245,66,0.06);">
    <div style="font-family:'Space Mono',monospace;font-size:11px;color:#c8f542;min-width:20px;">#${{i+1}}</div>
    <div style="flex:1;">
      <div style="font-size:11px;color:#e8eaf0;font-weight:600;">${{p.Name.length > 35 ? p.Name.slice(0,35)+'...' : p.Name}}</div>
      <div style="font-size:10px;color:#6b7280;margin-top:2px;">${{p.Brand}} · $${{p.Price}}</div>
      <div style="display:flex;flex-wrap:wrap;gap:3px;margin-top:4px;">
        ${{p.key_ingredients.map(ki => `<span style="padding:2px 6px;background:rgba(66,245,200,0.1);border:1px solid rgba(66,245,200,0.2);border-radius:4px;font-size:9px;color:#42f5c8;font-family:'Space Mono',monospace;">${{ki.ingredient}}</span>`).join('')}}
      </div>
    </div>
    <div style="text-align:center;">
      <div style="font-family:'Space Mono',monospace;font-size:18px;font-weight:700;color:#c8f542;">${{p.score}}</div>
      <div style="font-size:9px;color:#6b7280;">score</div>
    </div>
  </div>
`).join('');

// Heatmap
const heatmap = document.getElementById('heatmapContainer');
const brands = L2_DATA.brands;
const hdata = L2_DATA.heatmap;
const cellSize = Math.floor(280 / brands.length);
let hmHTML = `<div style="display:grid;grid-template-columns:80px ${{brands.map(()=>cellSize+'px').join(' ')}};gap:2px;font-family:'Space Mono',monospace;font-size:9px;">`;
hmHTML += `<div></div>${{brands.map(b=>`<div style="color:#6b7280;text-align:center;transform:rotate(-45deg);transform-origin:center;height:40px;display:flex;align-items:flex-end;justify-content:center;">${{b.slice(0,8)}}</div>`).join('')}}`;
brands.forEach((b,i) => {{
  hmHTML += `<div style="color:#6b7280;display:flex;align-items:center;">${{b.slice(0,10)}}</div>`;
  hdata[i].forEach((val,j) => {{
    const intensity = val/100;
    const r = Math.round(66 + (200-66)*intensity);
    const g = Math.round(245 + (66-245)*intensity);
    const color = `rgb(${{r}},${{g}},80)`;
    hmHTML += `<div title="${{brands[i]}} vs ${{brands[j]}}: ${{val}}%" style="width:${{cellSize}}px;height:${{cellSize}}px;background:${{color}};border-radius:3px;opacity:0.85;display:flex;align-items:center;justify-content:center;font-size:8px;color:#080b12;font-weight:700;">${{val > 10 ? Math.round(val)+'%' : ''}}</div>`;
  }});
}});
hmHTML += '</div>';
heatmap.innerHTML = hmHTML;

// Benefit Chart
const benefitCounts = L2_DATA.benefit_counts;
new Chart(document.getElementById('benefitChart'), {{
  type: 'bar',
  data: {{
    labels: Object.keys(benefitCounts),
    datasets: [{{
      data: Object.values(benefitCounts),
      backgroundColor: ['#42f5c8aa','#c8f542aa','#f542a0aa','#f5a742aa','#4287f5aa','#f54242aa','#42f587aa'],
      borderRadius: 4,
      borderSkipped: false,
    }}]
  }},
  options: {{
    responsive: true,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ grid: {{ display: false }}, ticks: {{ color: '#42f5c8', font: {{ size: 9 }} }} }},
      y: {{ grid: {{ color: 'rgba(200,245,66,0.06)' }}, ticks: {{ color: '#6b7280' }} }}
    }}
  }}
}});

// Ingredient Finder
document.getElementById('ingSearch').addEventListener('input', function() {{
  const query = this.value.toLowerCase().trim();
  const results = document.getElementById('ingResults');
  if (!query) {{ results.innerHTML = ''; return; }}
  const matches = PRODUCTS.filter(p => p.Ingredients.toLowerCase().includes(query)).slice(0,6);
  results.innerHTML = matches.length === 0
    ? `<div style="color:#6b7280;font-size:12px;font-family:'Space Mono',monospace;">No products found with "${{query}}"</div>`
    : matches.map(p => `
      <div style="padding:8px;background:#0e1220;border:1px solid rgba(200,245,66,0.1);border-radius:8px;">
        <div style="font-size:11px;color:#e8eaf0;font-weight:600;">${{p.Name.length>40?p.Name.slice(0,40)+'...':p.Name}}</div>
        <div style="font-size:10px;color:#6b7280;margin-top:2px;font-family:'Space Mono',monospace;">${{p.Brand}} · $${{p.Price}} · ★${{p.Rank}}</div>
      </div>`).join('');
}});
</script>
'''

html = html.replace('</body>', level2_section + '</body>')

with open('cosmetic_dashboard.html', 'w') as f:
    f.write(html)

print("✅ Level 2 added! Refresh your browser!")

✅ Level 2 added! Refresh your browser!


In [62]:
with open('cosmetic_dashboard.html', 'r') as f:
    html = f.read()

if 'INGREDIENT INTELLIGENCE' in html:
    print("✅ Level 2 is in the file! Scroll down in browser or do hard refresh")
    print("Try: Cmd + Shift + R to hard refresh the browser")
else:
    print("❌ Level 2 not found - need to re-run the injection code")

✅ Level 2 is in the file! Scroll down in browser or do hard refresh
Try: Cmd + Shift + R to hard refresh the browser


In [63]:
import json
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Build recommendation engine
# For each product, find top 5 most similar by ingredient cosine similarity
print("Building similarity matrix...")
similarity_matrix = cosine_similarity(A)

recommendations = []
for i, row in moisturizers_dry.iterrows():
    # Get top 5 similar products (excluding itself)
    sim_scores = list(enumerate(similarity_matrix[i]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != i][:5]
    
    recs = []
    for idx, score in sim_scores:
        p = moisturizers_dry.iloc[idx]
        recs.append({
            'Name': p['Name'],
            'Brand': p['Brand'],
            'Price': float(p['Price']),
            'Rank': float(p['Rank']),
            'similarity': round(float(score) * 100, 1)
        })
    recommendations.append(recs)

# Skin concern mapping
concern_ingredients = {
    'dryness': ['glycerin', 'hyaluronic acid', 'squalane', 'ceramide', 'shea butter'],
    'aging': ['retinol', 'peptide', 'collagen', 'niacinamide', 'vitamin c'],
    'oily': ['salicylic acid', 'zinc', 'niacinamide', 'clay'],
    'sensitive': ['aloe', 'ceramide', 'oat', 'centella', 'allantoin'],
    'brightening': ['vitamin c', 'niacinamide', 'kojic', 'arbutin', 'glycolic'],
    'pores': ['salicylic acid', 'niacinamide', 'retinol', 'clay', 'zinc']
}

rec_json = json.dumps({
    'recommendations': recommendations,
    'concern_ingredients': concern_ingredients
})

print(f"✅ Similarity matrix built! Shape: {similarity_matrix.shape}")
print(f"Sample recommendation for product 0:")
print(f"  {moisturizers_dry.iloc[0]['Name']}")
print(f"  → Similar to: {recommendations[0][0]['Name']} ({recommendations[0][0]['similarity']}% match)")

Building similarity matrix...
✅ Similarity matrix built! Shape: (190, 190)
Sample recommendation for product 0:
  Crème de la Mer
  → Similar to: Crème de la Mer Mini (100.0% match)


In [64]:
with open('cosmetic_dashboard.html', 'r') as f:
    html = f.read()

level3_section = f'''
<div style="position:relative;z-index:1;background:#0e1220;border-top:1px solid rgba(245,66,160,0.2);padding:24px 32px 60px;">
  <div style="display:flex;align-items:center;gap:12px;margin-bottom:24px;">
    <span style="font-family:'Space Mono',monospace;font-size:10px;letter-spacing:2px;color:#f542a0;">// RECOMMENDATION ENGINE</span>
    <div style="flex:1;height:1px;background:rgba(245,66,160,0.15);"></div>
  </div>

  <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">

    <!-- SKIN CONCERN RECOMMENDER -->
    <div style="background:#141824;border:1px solid rgba(245,66,160,0.15);border-radius:12px;padding:24px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#6b7280;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:16px;">🎯 Get Personalized Recommendations</div>

      <div style="margin-bottom:12px;">
        <div style="font-size:11px;color:#6b7280;margin-bottom:8px;font-family:'Space Mono',monospace;">YOUR SKIN CONCERNS</div>
        <div style="display:flex;flex-wrap:wrap;gap:6px;" id="concernChips">
          <div class="concern-chip" onclick="toggleConcern(this,'dryness')" style="padding:6px 14px;border-radius:20px;font-size:11px;border:1px solid rgba(245,66,160,0.3);background:transparent;color:#6b7280;cursor:pointer;transition:all 0.2s;font-family:'Space Mono',monospace;">💧 Dryness</div>
          <div class="concern-chip" onclick="toggleConcern(this,'aging')" style="padding:6px 14px;border-radius:20px;font-size:11px;border:1px solid rgba(245,66,160,0.3);background:transparent;color:#6b7280;cursor:pointer;transition:all 0.2s;font-family:'Space Mono',monospace;">⏳ Aging</div>
          <div class="concern-chip" onclick="toggleConcern(this,'oily')" style="padding:6px 14px;border-radius:20px;font-size:11px;border:1px solid rgba(245,66,160,0.3);background:transparent;color:#6b7280;cursor:pointer;transition:all 0.2s;font-family:'Space Mono',monospace;">🧴 Oily Skin</div>
          <div class="concern-chip" onclick="toggleConcern(this,'sensitive')" style="padding:6px 14px;border-radius:20px;font-size:11px;border:1px solid rgba(245,66,160,0.3);background:transparent;color:#6b7280;cursor:pointer;transition:all 0.2s;font-family:'Space Mono',monospace;">🌿 Sensitive</div>
          <div class="concern-chip" onclick="toggleConcern(this,'brightening')" style="padding:6px 14px;border-radius:20px;font-size:11px;border:1px solid rgba(245,66,160,0.3);background:transparent;color:#6b7280;cursor:pointer;transition:all 0.2s;font-family:'Space Mono',monospace;">✨ Brightening</div>
          <div class="concern-chip" onclick="toggleConcern(this,'pores')" style="padding:6px 14px;border-radius:20px;font-size:11px;border:1px solid rgba(245,66,160,0.3);background:transparent;color:#6b7280;cursor:pointer;transition:all 0.2s;font-family:'Space Mono',monospace;">🔬 Pores</div>
        </div>
      </div>

      <div style="margin-bottom:12px;">
        <div style="font-size:11px;color:#6b7280;margin-bottom:8px;font-family:'Space Mono',monospace;">MAX BUDGET</div>
        <div style="display:flex;align-items:center;gap:12px;">
          <input type="range" id="budgetSlider" min="10" max="300" value="100" step="10"
            style="flex:1;-webkit-appearance:none;height:3px;background:linear-gradient(to right,#f542a0 0%,#f542a0 33%,#141824 33%);border-radius:2px;outline:none;cursor:pointer;"
            oninput="updateBudget(this)">
          <span id="budgetLabel" style="font-family:'Space Mono',monospace;font-size:13px;color:#f542a0;min-width:50px;">$100</span>
        </div>
      </div>

      <button onclick="getRecommendations()" style="width:100%;padding:12px;background:linear-gradient(135deg,#f542a0,#a042f5);border:none;border-radius:8px;color:#fff;font-family:'Space Mono',monospace;font-size:12px;font-weight:700;cursor:pointer;letter-spacing:1px;margin-top:4px;">
        ✨ GET MY RECOMMENDATIONS
      </button>

      <div id="recResults" style="margin-top:16px;display:flex;flex-direction:column;gap:8px;"></div>
    </div>

    <!-- PRODUCT SIMILARITY FINDER -->
    <div style="background:#141824;border:1px solid rgba(245,66,160,0.15);border-radius:12px;padding:24px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#6b7280;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:16px;">🔄 Find Similar Products (Dupes Finder)</div>

      <div style="margin-bottom:12px;">
        <div style="font-size:11px;color:#6b7280;margin-bottom:8px;font-family:'Space Mono',monospace;">SELECT A PRODUCT</div>
        <input type="text" id="dupeSearch" placeholder="Type product name..."
          style="width:100%;background:#0e1220;border:1px solid rgba(245,66,160,0.2);border-radius:8px;padding:10px 14px;color:#e8eaf0;font-family:'Space Mono',monospace;font-size:12px;outline:none;margin-bottom:8px;"
          oninput="searchProducts(this.value)">
        <div id="productDropdown" style="display:flex;flex-direction:column;gap:4px;max-height:150px;overflow-y:auto;"></div>
      </div>

      <div id="dupeResults" style="margin-top:12px;">
        <div style="text-align:center;padding:30px;color:#6b7280;font-family:'Space Mono',monospace;font-size:11px;">
          Search for a product to find its dupes
        </div>
      </div>
    </div>

  </div>
</div>

<script>
const REC_DATA = {rec_json};
let selectedConcerns = new Set();
let maxBudget = 100;
let selectedProductIdx = -1;

function toggleConcern(el, concern) {{
  if(selectedConcerns.has(concern)) {{
    selectedConcerns.delete(concern);
    el.style.background = 'transparent';
    el.style.color = '#6b7280';
    el.style.borderColor = 'rgba(245,66,160,0.3)';
  }} else {{
    selectedConcerns.add(concern);
    el.style.background = '#f542a0';
    el.style.color = '#fff';
    el.style.borderColor = '#f542a0';
  }}
}}

function updateBudget(el) {{
  maxBudget = parseInt(el.value);
  const pct = (maxBudget - 10) / 290 * 100;
  el.style.background = `linear-gradient(to right,#f542a0 0%,#f542a0 ${{pct}}%,#0e1220 ${{pct}}%)`;
  document.getElementById('budgetLabel').textContent = '$' + maxBudget;
}}

function getRecommendations() {{
  const results = document.getElementById('recResults');
  if(selectedConcerns.size === 0) {{
    results.innerHTML = '<div style="color:#f542a0;font-family:\\'Space Mono\\',monospace;font-size:11px;text-align:center;padding:12px;">Please select at least one skin concern!</div>';
    return;
  }}

  const targetIngs = new Set();
  selectedConcerns.forEach(c => {{
    REC_DATA.concern_ingredients[c].forEach(i => targetIngs.add(i));
  }});

  const scored = PRODUCTS
    .filter(p => p.Price <= maxBudget)
    .map(p => {{
      const ingLower = p.Ingredients.toLowerCase();
      let score = 0;
      targetIngs.forEach(ing => {{ if(ingLower.includes(ing)) score++; }});
      return {{...p, recScore: score}};
    }})
    .filter(p => p.recScore > 0)
    .sort((a,b) => b.recScore - a.recScore || b.Rank - a.Rank)
    .slice(0, 6);

  if(scored.length === 0) {{
    results.innerHTML = '<div style="color:#6b7280;font-family:\\'Space Mono\\',monospace;font-size:11px;text-align:center;padding:12px;">No products found. Try increasing your budget!</div>';
    return;
  }}

  results.innerHTML = scored.map((p,i) => `
    <div style="padding:10px 12px;background:#0e1220;border:1px solid rgba(245,66,160,0.15);border-radius:8px;display:flex;align-items:center;gap:10px;">
      <div style="font-family:'Space Mono',monospace;font-size:13px;color:#f542a0;font-weight:700;min-width:24px;">#${{i+1}}</div>
      <div style="flex:1;">
        <div style="font-size:11px;color:#e8eaf0;font-weight:600;">${{p.Name.length>40?p.Name.slice(0,40)+'...':p.Name}}</div>
        <div style="font-size:10px;color:#6b7280;font-family:'Space Mono',monospace;margin-top:2px;">${{p.Brand}} · $${{p.Price}} · ★${{p.Rank}}</div>
      </div>
      <div style="text-align:center;">
        <div style="font-family:'Space Mono',monospace;font-size:16px;font-weight:700;color:#f542a0;">${{p.recScore}}</div>
        <div style="font-size:9px;color:#6b7280;">matches</div>
      </div>
    </div>
  `).join('');
}}

function searchProducts(query) {{
  const dropdown = document.getElementById('productDropdown');
  if(!query || query.length < 2) {{ dropdown.innerHTML = ''; return; }}
  const matches = PRODUCTS.map((p,i) => ({{...p, idx:i}}))
    .filter(p => p.Name.toLowerCase().includes(query.toLowerCase()))
    .slice(0, 6);
  dropdown.innerHTML = matches.map(p => `
    <div onclick="selectProductForDupe(${{p.idx}},'${{p.Name.replace(/'/g,"\\'")}}',this)"
      style="padding:8px 10px;background:#0e1220;border:1px solid rgba(245,66,160,0.1);border-radius:6px;cursor:pointer;font-size:11px;color:#e8eaf0;transition:all 0.2s;"
      onmouseover="this.style.borderColor='#f542a0'" onmouseout="this.style.borderColor='rgba(245,66,160,0.1)'">
      <div style="font-weight:600;">${{p.Name.length>45?p.Name.slice(0,45)+'...':p.Name}}</div>
      <div style="font-size:10px;color:#6b7280;font-family:'Space Mono',monospace;">${{p.Brand}} · $${{p.Price}}</div>
    </div>
  `).join('');
}}

function selectProductForDupe(idx, name, el) {{
  selectedProductIdx = idx;
  document.getElementById('dupeSearch').value = name;
  document.getElementById('productDropdown').innerHTML = '';

  const recs = REC_DATA.recommendations[idx];
  document.getElementById('dupeResults').innerHTML = `
    <div style="font-family:'Space Mono',monospace;font-size:10px;color:#f542a0;letter-spacing:1px;margin-bottom:10px;">SIMILAR PRODUCTS (DUPES)</div>
    ${{recs.map((r,i) => `
      <div style="padding:10px 12px;background:#0e1220;border:1px solid rgba(245,66,160,0.12);border-radius:8px;display:flex;align-items:center;gap:10px;margin-bottom:6px;">
        <div style="flex:1;">
          <div style="font-size:11px;color:#e8eaf0;font-weight:600;">${{r.Name.length>40?r.Name.slice(0,40)+'...':r.Name}}</div>
          <div style="font-size:10px;color:#6b7280;font-family:'Space Mono',monospace;margin-top:2px;">${{r.Brand}} · $${{r.Price}}</div>
        </div>
        <div style="text-align:center;">
          <div style="font-family:'Space Mono',monospace;font-size:14px;font-weight:700;color:${{r.similarity>50?'#c8f542':'#f5a742'}}">${{r.similarity}}%</div>
          <div style="font-size:9px;color:#6b7280;">match</div>
        </div>
      </div>
    `).join('')}}
  `;
}}
</script>
'''

html = html.replace('</body>', level3_section + '</body>')

with open('cosmetic_dashboard.html', 'w') as f:
    f.write(html)

print("✅ Level 3 Recommendation Engine added!")
print("Refresh your browser and scroll to the bottom!")

✅ Level 3 Recommendation Engine added!
Refresh your browser and scroll to the bottom!


In [65]:
with open('cosmetic_dashboard.html', 'r') as f:
    html = f.read()

about_section = '''
<div style="position:relative;z-index:1;background:#080b12;border-top:1px solid rgba(200,245,66,0.15);padding:40px 32px 80px;">
  <div style="display:flex;align-items:center;gap:12px;margin-bottom:32px;">
    <span style="font-family:'Space Mono',monospace;font-size:10px;letter-spacing:2px;color:#c8f542;">// ABOUT THIS PROJECT</span>
    <div style="flex:1;height:1px;background:rgba(200,245,66,0.1);"></div>
  </div>

  <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:20px;margin-bottom:32px;">

    <!-- About Card -->
    <div style="background:#141824;border:1px solid rgba(200,245,66,0.12);border-radius:12px;padding:24px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#c8f542;letter-spacing:1.5px;margin-bottom:12px;">👩‍💻 BUILT BY</div>
      <div style="font-size:22px;font-weight:800;color:#e8eaf0;margin-bottom:4px;">Aiswarya</div>
      <div style="font-size:12px;color:#6b7280;font-family:'Space Mono',monospace;margin-bottom:16px;">Data Analytics Portfolio Project</div>
      <div style="display:flex;flex-direction:column;gap:6px;">
        <div style="display:flex;align-items:center;gap:8px;font-size:11px;color:#6b7280;">
          <span style="color:#c8f542;">→</span> Machine Learning · t-SNE
        </div>
        <div style="display:flex;align-items:center;gap:8px;font-size:11px;color:#6b7280;">
          <span style="color:#c8f542;">→</span> NLP · Ingredient Tokenization
        </div>
        <div style="display:flex;align-items:center;gap:8px;font-size:11px;color:#6b7280;">
          <span style="color:#c8f542;">→</span> Cosine Similarity · Recommendations
        </div>
        <div style="display:flex;align-items:center;gap:8px;font-size:11px;color:#6b7280;">
          <span style="color:#c8f542;">→</span> Data Visualization · Analytics
        </div>
        <div style="display:flex;align-items:center;gap:8px;font-size:11px;color:#6b7280;">
          <span style="color:#c8f542;">→</span> Google Cloud · Deployment
        </div>
      </div>
    </div>

    <!-- Tech Stack -->
    <div style="background:#141824;border:1px solid rgba(200,245,66,0.12);border-radius:12px;padding:24px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#42f5c8;letter-spacing:1.5px;margin-bottom:12px;">🛠️ TECH STACK</div>
      <div style="display:flex;flex-direction:column;gap:8px;">
        <div style="display:flex;align-items:center;justify-content:space-between;">
          <span style="font-size:12px;color:#e8eaf0;">Python</span>
          <div style="display:flex;gap:3px;">
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
          </div>
        </div>
        <div style="display:flex;align-items:center;justify-content:space-between;">
          <span style="font-size:12px;color:#e8eaf0;">Pandas & NumPy</span>
          <div style="display:flex;gap:3px;">
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
          </div>
        </div>
        <div style="display:flex;align-items:center;justify-content:space-between;">
          <span style="font-size:12px;color:#e8eaf0;">Scikit-learn</span>
          <div style="display:flex;gap:3px;">
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#6b7280;"></div>
          </div>
        </div>
        <div style="display:flex;align-items:center;justify-content:space-between;">
          <span style="font-size:12px;color:#e8eaf0;">Chart.js</span>
          <div style="display:flex;gap:3px;">
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#6b7280;"></div>
          </div>
        </div>
        <div style="display:flex;align-items:center;justify-content:space-between;">
          <span style="font-size:12px;color:#e8eaf0;">Google Cloud</span>
          <div style="display:flex;gap:3px;">
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#6b7280;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#6b7280;"></div>
          </div>
        </div>
        <div style="display:flex;align-items:center;justify-content:space-between;">
          <span style="font-size:12px;color:#e8eaf0;">HTML/CSS/JS</span>
          <div style="display:flex;gap:3px;">
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#42f5c8;"></div>
          </div>
        </div>
      </div>
    </div>

    <!-- Project Stats -->
    <div style="background:#141824;border:1px solid rgba(200,245,66,0.12);border-radius:12px;padding:24px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#f542a0;letter-spacing:1.5px;margin-bottom:12px;">📊 PROJECT STATS</div>
      <div style="display:flex;flex-direction:column;gap:12px;">
        <div style="display:flex;justify-content:space-between;align-items:center;padding-bottom:10px;border-bottom:1px solid rgba(200,245,66,0.06);">
          <span style="font-size:11px;color:#6b7280;">Dataset</span>
          <span style="font-family:'Space Mono',monospace;font-size:12px;color:#c8f542;">1,472 products</span>
        </div>
        <div style="display:flex;justify-content:space-between;align-items:center;padding-bottom:10px;border-bottom:1px solid rgba(200,245,66,0.06);">
          <span style="font-size:11px;color:#6b7280;">Analyzed</span>
          <span style="font-family:'Space Mono',monospace;font-size:12px;color:#c8f542;">190 moisturizers</span>
        </div>
        <div style="display:flex;justify-content:space-between;align-items:center;padding-bottom:10px;border-bottom:1px solid rgba(200,245,66,0.06);">
          <span style="font-size:11px;color:#6b7280;">Unique ingredients</span>
          <span style="font-family:'Space Mono',monospace;font-size:12px;color:#42f5c8;">2,233</span>
        </div>
        <div style="display:flex;justify-content:space-between;align-items:center;padding-bottom:10px;border-bottom:1px solid rgba(200,245,66,0.06);">
          <span style="font-size:11px;color:#6b7280;">Brands covered</span>
          <span style="font-family:'Space Mono',monospace;font-size:12px;color:#42f5c8;">53 brands</span>
        </div>
        <div style="display:flex;justify-content:space-between;align-items:center;padding-bottom:10px;border-bottom:1px solid rgba(200,245,66,0.06);">
          <span style="font-size:11px;color:#6b7280;">ML Algorithm</span>
          <span style="font-family:'Space Mono',monospace;font-size:12px;color:#f542a0;">t-SNE + Cosine</span>
        </div>
        <div style="display:flex;justify-content:space-between;align-items:center;">
          <span style="font-size:11px;color:#6b7280;">Deployed on</span>
          <span style="font-family:'Space Mono',monospace;font-size:12px;color:#f542a0;">Google Cloud</span>
        </div>
      </div>
    </div>

  </div>

  <!-- Footer -->
  <div style="text-align:center;padding-top:24px;border-top:1px solid rgba(200,245,66,0.08);">
    <div style="font-family:'Space Mono',monospace;font-size:11px;color:#6b7280;">
      Built with 💚 by <span style="color:#c8f542;">Aiswarya</span> · 
      Data from <span style="color:#42f5c8;">Sephora via Kaggle</span> · 
      Powered by <span style="color:#f542a0;">Python + ML + Google Cloud</span>
    </div>
  </div>
</div>
'''

html = html.replace('</body>', about_section + '</body>')

with open('cosmetic_dashboard.html', 'w') as f:
    f.write(html)

print("✅ Level 4 About section added!")
print("Now let's deploy the final version to Google Cloud!")

✅ Level 4 About section added!
Now let's deploy the final version to Google Cloud!


In [66]:
with open('cosmetic_dashboard.html', 'r') as f:
    html = f.read()

# Lavender Dreams theme replacements
theme_replacements = [
    # Backgrounds
    ('--bg:#0d0510', '--bg:#130d1a'),
    ('--bg:#080b12', '--bg:#130d1a'),
    ('--bg2:#1a0c1f', '--bg2:#1e1228'),
    ('--bg2:#0e1220', '--bg2:#1e1228'),
    ('--bg3:#220f28', '--bg3:#261530'),
    ('--bg3:#141824', '--bg3:#261530'),
    # Accent colors
    ('--accent:#ff85c2', '--accent:#c9a0dc'),
    ('--accent:#c8f542', '--accent:#c9a0dc'),
    ('--accent2:#ffaad4', '--accent2:#e8d0f5'),
    ('--accent2:#42f5c8', '--accent2:#e8d0f5'),
    ('--accent3:#cc5599', '--accent3:#f4c2ce'),
    ('--accent3:#f542a0', '--accent3:#f4c2ce'),
    # Text
    ('--text:#ffe0f0', '--text:#f0e0ff'),
    ('--text:#e8eaf0', '--text:#f0e0ff'),
    # Muted
    ('--muted:#994477', '--muted:#8060a0'),
    ('--muted:#6b7280', '--muted:#8060a0'),
    # Border & glow
    ('--border:rgba(255,133,194,0.2)', '--border:rgba(201,160,220,0.2)'),
    ('--border:rgba(200,245,66,0.15)', '--border:rgba(201,160,220,0.2)'),
    ('--glow:rgba(255,133,194,0.35)', '--glow:rgba(201,160,220,0.35)'),
    ('--glow:rgba(200,245,66,0.3)', '--glow:rgba(201,160,220,0.35)'),
    # Accent colors in HTML
    ('color:#ff85c2', 'color:#c9a0dc'),
    ('color:#c8f542', 'color:#c9a0dc'),
    ('color:#ffaad4', 'color:#e8d0f5'),
    ('color:#42f5c8', 'color:#e8d0f5'),
    ('color:#cc5599', 'color:#f4c2ce'),
    ('color:#f542a0', 'color:#f4c2ce'),
    # Grid background dots
    ('rgba(255,133,194,0.04)', 'rgba(201,160,220,0.04)'),
    ('rgba(200,245,66,0.03)', 'rgba(201,160,220,0.04)'),
    # Active chips
    ('background:#ff85c2;border-color:#ff85c2;color:#0d0510',
     'background:#c9a0dc;border-color:#c9a0dc;color:#130d1a'),
    ('background:var(--accent);border-color:var(--accent);color:#080b12',
     'background:#c9a0dc;border-color:#c9a0dc;color:#130d1a'),
    # Slider thumb
    ('background:#ff85c2;box-shadow:0 0 8px rgba(255,133,194,0.4)',
     'background:#c9a0dc;box-shadow:0 0 8px rgba(201,160,220,0.4)'),
    # Bottom bar dot
    ('background:#ff85c2;box-shadow:0 0 6px #ff85c2',
     'background:#c9a0dc;box-shadow:0 0 6px #c9a0dc'),
    # Tooltip border
    ('border:1px solid #ff85c2', 'border:1px solid #c9a0dc'),
    # Chart borders
    ('rgba(255,133,194,0.15)', 'rgba(201,160,220,0.15)'),
    ('rgba(255,170,212,0.15)', 'rgba(232,208,245,0.12)'),
    ('rgba(204,85,153,0.2)', 'rgba(244,194,206,0.15)'),
    # Recommendation button gradient
    ('background:linear-gradient(135deg,#f542a0,#a042f5)',
     'background:linear-gradient(135deg,#c9a0dc,#9b7fd4)'),
    ('background:linear-gradient(135deg,#ff85c2,#cc5599)',
     'background:linear-gradient(135deg,#c9a0dc,#9b7fd4)'),
    # Border colors in sections
    ('rgba(200,245,66,0.1)', 'rgba(201,160,220,0.1)'),
    ('rgba(66,245,200,0.1)', 'rgba(232,208,245,0.1)'),
    ('rgba(245,66,160,0.2)', 'rgba(201,160,220,0.15)'),
    # Stat borders
    ('rgba(200,245,66,0.06)', 'rgba(201,160,220,0.08)'),
    ('rgba(200,245,66,0.08)', 'rgba(201,160,220,0.08)'),
    # Footer
    ("color:#ff85c2;'>Aiswarya", "color:#c9a0dc;'>Aiswarya"),
    ("color:#ffaad4;'>Sephora", "color:#e8d0f5;'>Sephora"),
    ("color:#cc5599;'>Python", "color:#f4c2ce;'>Python"),
    ("color:#c8f542;'>Aiswarya", "color:#c9a0dc;'>Aiswarya"),
    ("color:#42f5c8;'>Sephora", "color:#e8d0f5;'>Sephora"),
]

for old, new in theme_replacements:
    html = html.replace(old, new)

# Update font to elegant serif
html = html.replace(
    "family=Space+Mono:wght@400;700&family=Playfair+Display:wght@400;700;900&family=DM+Sans:wght@400;600;800",
    "family=Space+Mono:wght@400;700&family=Cormorant+Garamond:wght@400;600;700&family=Jost:wght@300;400;600;800"
)
html = html.replace("'DM Sans', sans-serif", "'Jost', sans-serif")
html = html.replace("'Syne', sans-serif", "'Jost', sans-serif")

# Update logo
html = html.replace(
    '<div class="logo-text" style="font-family:\'Playfair Display\',serif;font-size:20px;">Cosmetic<span style="color:#ff85c2;">Lab</span> 💗</div>',
    '<div class="logo-text" style="font-family:\'Cormorant Garamond\',serif;font-size:22px;letter-spacing:1px;">Cosmetic<span style="color:#c9a0dc;">Lab</span> ✦</div>'
)
html = html.replace(
    '<div class="logo-text">Cosmetic<span>Lab</span></div>',
    '<div class="logo-text" style="font-family:\'Cormorant Garamond\',serif;font-size:22px;letter-spacing:1px;">Cosmetic<span style="color:#c9a0dc;">Lab</span> ✦</div>'
)

# Update logo icon emoji
html = html.replace(
    '<div class="logo-icon">🧬</div>',
    '<div class="logo-icon" style="background:#c9a0dc;font-size:18px;">✦</div>'
)

# Update title
html = html.replace(
    '<title>💗 CosmeticLab — Bubblegum Pop</title>',
    '<title>✦ CosmeticLab — Lavender Dreams</title>'
)
html = html.replace(
    '<title>Cosmetic Intelligence Dashboard</title>',
    '<title>✦ CosmeticLab — Lavender Dreams</title>'
)

with open('cosmetic_dashboard.html', 'w') as f:
    f.write(html)

print("✅ Lavender Dreams theme applied! 💜")
print("Refresh your browser to see it!")

✅ Lavender Dreams theme applied! 💜
Refresh your browser to see it!


In [67]:
with open('cosmetic_dashboard.html', 'r') as f:
    html = f.read()

# 1. ADD LOADING SCREEN
loading_screen = '''
<div id="loadingScreen" style="position:fixed;inset:0;background:#130d1a;z-index:9999;display:flex;flex-direction:column;align-items:center;justify-content:center;transition:opacity 0.8s ease;">
  <div style="font-family:'Cormorant Garamond',serif;font-size:48px;font-weight:700;color:#c9a0dc;letter-spacing:4px;margin-bottom:8px;animation:fadeInUp 1s ease;">
    Cosmetic<span style="color:#e8d0f5;">Lab</span> ✦
  </div>
  <div style="font-family:'Space Mono',monospace;font-size:11px;color:#8060a0;letter-spacing:3px;margin-bottom:40px;animation:fadeInUp 1s ease 0.3s both;">
    INGREDIENT INTELLIGENCE DASHBOARD
  </div>
  <div style="display:flex;gap:8px;">
    <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;animation:bounce 1.2s ease infinite;"></div>
    <div style="width:8px;height:8px;border-radius:50%;background:#e8d0f5;animation:bounce 1.2s ease infinite 0.2s;"></div>
    <div style="width:8px;height:8px;border-radius:50%;background:#f4c2ce;animation:bounce 1.2s ease infinite 0.4s;"></div>
  </div>
  <style>
    @keyframes fadeInUp { from { opacity:0; transform:translateY(20px); } to { opacity:1; transform:translateY(0); } }
    @keyframes bounce { 0%,100% { transform:translateY(0); opacity:0.4; } 50% { transform:translateY(-10px); opacity:1; } }
  </style>
</div>
<script>
  window.addEventListener('load', () => {
    setTimeout(() => {
      const ls = document.getElementById('loadingScreen');
      ls.style.opacity = '0';
      setTimeout(() => ls.style.display = 'none', 800);
    }, 2000);
  });
</script>
'''

# 2. ADD SPARKLE PARTICLES BACKGROUND
sparkles = '''
<canvas id="sparkleCanvas" style="position:fixed;inset:0;pointer-events:none;z-index:0;opacity:0.4;"></canvas>
<script>
(function() {
  const canvas = document.getElementById('sparkleCanvas');
  const ctx = canvas.getContext('2d');
  canvas.width = window.innerWidth;
  canvas.height = window.innerHeight;
  window.addEventListener('resize', () => { canvas.width = window.innerWidth; canvas.height = window.innerHeight; });

  const particles = Array.from({length: 60}, () => ({
    x: Math.random() * canvas.width,
    y: Math.random() * canvas.height,
    r: Math.random() * 2 + 0.5,
    dx: (Math.random() - 0.5) * 0.3,
    dy: (Math.random() - 0.5) * 0.3,
    opacity: Math.random(),
    dop: (Math.random() - 0.5) * 0.01,
    color: ['#c9a0dc','#e8d0f5','#f4c2ce','#9b7fd4'][Math.floor(Math.random()*4)]
  }));

  function animateParticles() {
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    particles.forEach(p => {
      p.x += p.dx; p.y += p.dy;
      p.opacity += p.dop;
      if (p.opacity <= 0 || p.opacity >= 1) p.dop *= -1;
      if (p.x < 0 || p.x > canvas.width) p.dx *= -1;
      if (p.y < 0 || p.y > canvas.height) p.dy *= -1;
      ctx.beginPath();
      ctx.arc(p.x, p.y, p.r, 0, Math.PI * 2);
      ctx.fillStyle = p.color;
      ctx.globalAlpha = p.opacity;
      ctx.fill();
    });
    ctx.globalAlpha = 1;
    requestAnimationFrame(animateParticles);
  }
  animateParticles();
})();
</script>
'''

# 3. ADD SHARE + DOWNLOAD BUTTONS to toolbar
old_toolbar = '<button class="viz-btn" onclick="resetZoom()">Reset View</button>'
new_toolbar = '''<button class="viz-btn" onclick="resetZoom()">Reset View</button>
      <button class="viz-btn" onclick="downloadChart()" title="Download chart as PNG">⬇ Download</button>
      <button class="viz-btn" onclick="shareURL()" title="Copy live URL">🔗 Share</button>'''

# 4. ADD DOWNLOAD + SHARE FUNCTIONS
download_share_script = '''
<script>
function downloadChart() {
  const canvas = document.getElementById('scatterCanvas');
  const link = document.createElement('a');
  link.download = 'cosmetic_ingredient_map.png';
  link.href = canvas.toDataURL('image/png');
  link.click();
}

function shareURL() {
  const url = 'https://storage.googleapis.com/cosmetic-map-aishu/cosmetic_dashboard.html';
  navigator.clipboard.writeText(url).then(() => {
    const btn = document.querySelector('[onclick="shareURL()"]');
    const orig = btn.innerHTML;
    btn.innerHTML = '✅ Copied!';
    btn.style.color = '#c9a0dc';
    setTimeout(() => { btn.innerHTML = orig; btn.style.color = ''; }, 2000);
  });
}
</script>
'''

# 5. UPDATE ABOUT SECTION with LinkedIn + GitHub
old_about_footer = 'Built with 💚 by <span style="color:#c9a0dc;">Aiswarya</span>'
new_about_footer = '''Built with 💜 by <span style="color:#c9a0dc;">Aiswarya</span> &nbsp;·&nbsp;
      <a href="https://www.linkedin.com/in/aiswarya-" target="_blank" 
        style="color:#e8d0f5;text-decoration:none;border-bottom:1px solid rgba(232,208,245,0.3);padding-bottom:1px;transition:all 0.2s;"
        onmouseover="this.style.color='#c9a0dc'" onmouseout="this.style.color='#e8d0f5'">
        LinkedIn ↗
      </a> &nbsp;·&nbsp;
      <a href="https://github.com/Aiswarya0109" target="_blank"
        style="color:#f4c2ce;text-decoration:none;border-bottom:1px solid rgba(244,194,206,0.3);padding-bottom:1px;transition:all 0.2s;"
        onmouseover="this.style.color='#c9a0dc'" onmouseout="this.style.color='#f4c2ce'">
        GitHub ↗
      </a>'''

# 6. ADD MOBILE RESPONSIVE CSS
mobile_css = '''
<style>
@media (max-width: 768px) {
  .layout {
    grid-template-columns: 1fr !important;
    grid-template-rows: auto auto auto !important;
    height: auto !important;
  }
  .left-panel {
    border-right: none !important;
    border-bottom: 1px solid rgba(201,160,220,0.15);
    max-height: 300px;
  }
  .right-panel {
    border-left: none !important;
    border-top: 1px solid rgba(201,160,220,0.15);
  }
  canvas#scatterCanvas {
    height: 400px !important;
  }
  .header-stats {
    gap: 16px !important;
  }
  .hstat-num { font-size: 16px !important; }
  .hstat-label { font-size: 9px !important; }
  .logo-text { font-size: 16px !important; }
  #chartsPanel > div > div {
    grid-template-columns: 1fr 1fr !important;
  }
}
@media (max-width: 480px) {
  header { padding: 12px 16px !important; }
  .header-stats { gap: 10px !important; }
  #chartsPanel > div > div {
    grid-template-columns: 1fr !important;
  }
  .layout > div { padding: 12px !important; }
}
</style>
'''

# 7. ADD DOT ANIMATION on first load
dot_animation = '''
<script>
let dotsAnimated = false;
const origDrawCanvas = drawCanvas;
function drawCanvas() {
  if (!dotsAnimated && filteredIndices.length > 0) {
    dotsAnimated = true;
    let progress = 0;
    const total = filteredIndices.length;
    const interval = setInterval(() => {
      progress = Math.min(progress + 5, total);
      const subset = filteredIndices.slice(0, progress);
      const orig = filteredIndices;
      filteredIndices = subset;
      origDrawCanvas();
      filteredIndices = orig;
      if (progress >= total) {
        clearInterval(interval);
        origDrawCanvas();
      }
    }, 30);
  } else {
    origDrawCanvas();
  }
}
</script>
'''

# Apply all changes
html = loading_screen + html
html = html.replace('<canvas id="sparkleCanvas"', '<!-- sparkle exists --><canvas id="sparkleCanvas"') if 'sparkleCanvas' in html else html.replace('<div class="layout">', sparkles + '<div class="layout">')
html = html.replace(old_toolbar, new_toolbar)
html = html.replace(old_about_footer, new_about_footer)
html = html.replace('</head>', mobile_css + '</head>')
html = html.replace('</body>', download_share_script + dot_animation + '</body>')

with open('cosmetic_dashboard.html', 'w') as f:
    f.write(html)

print("✅ ALL UPGRADES APPLIED!")
print("✨ Loading screen")
print("💫 Sparkle particles")
print("🎬 Dot animations")
print("📱 Mobile responsive")
print("⬇️  Download PNG button")
print("🔗 Share button")
print("💼 LinkedIn & GitHub links")
print("\nRefresh your browser to see everything!")

✅ ALL UPGRADES APPLIED!
✨ Loading screen
💫 Sparkle particles
🎬 Dot animations
📱 Mobile responsive
⬇️  Download PNG button
🔗 Share button
💼 LinkedIn & GitHub links

Refresh your browser to see everything!


In [68]:
with open('cosmetic_dashboard.html', 'r') as f:
    html = f.read()

# Fix 1: Remove the broken dot animation override that's causing issues
html = html.replace('''<script>
let dotsAnimated = false;
const origDrawCanvas = drawCanvas;
function drawCanvas() {
  if (!dotsAnimated && filteredIndices.length > 0) {
    dotsAnimated = true;
    let progress = 0;
    const total = filteredIndices.length;
    const interval = setInterval(() => {
      progress = Math.min(progress + 5, total);
      const subset = filteredIndices.slice(0, progress);
      const orig = filteredIndices;
      filteredIndices = subset;
      origDrawCanvas();
      filteredIndices = orig;
      if (progress >= total) {
        clearInterval(interval);
        origDrawCanvas();
      }
    }, 30);
  } else {
    origDrawCanvas();
  }
}
</script>''', '')

# Fix 2: Make dots more visible with stronger colors
html = html.replace(
    "ctx.fillStyle=isSel?color:color+'bb'",
    "ctx.fillStyle=isSel?color:color+'ee'"
)
html = html.replace(
    "ctx.fillStyle='rgba(100,110,130,0.12)'",
    "ctx.fillStyle='rgba(150,100,180,0.2)'"
)

with open('cosmetic_dashboard.html', 'w') as f:
    f.write(html)

print("✅ Fixed! Refresh your browser!")

✅ Fixed! Refresh your browser!


In [69]:
# Restart everything fresh - rebuild the dashboard cleanly
import json
import re

# Get all data fresh
data = moisturizers_dry[['Name','Brand','Price','Rank','Ingredients',
                          'Dry','Normal','Oily','Combination','Sensitive',
                          'X','Y']].to_dict('records')
for p in data:
    for k in ['Price','Rank','X','Y']:
        p[k] = float(p[k])
    for k in ['Dry','Normal','Oily','Combination','Sensitive']:
        p[k] = int(p[k])

data_json = json.dumps(data, ensure_ascii=False)

print(f"Data ready: {len(data)} products")
print("✅ Now run the next cell!")

Data ready: 190 products
✅ Now run the next cell!


In [70]:
dashboard = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>CosmeticLab - Lavender Dreams</title>
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Cormorant+Garamond:wght@400;600;700&family=Jost:wght@300;400;600;800&display=swap" rel="stylesheet">
<style>
:root{{
  --bg:#130d1a;--bg2:#1e1228;--bg3:#261530;
  --accent:#c9a0dc;--accent2:#e8d0f5;--accent3:#f4c2ce;
  --text:#f0e0ff;--muted:#8060a0;
  --border:rgba(201,160,220,0.2);--glow:rgba(201,160,220,0.35);
}}
*{{margin:0;padding:0;box-sizing:border-box;}}
body{{background:var(--bg);color:var(--text);font-family:'Jost',sans-serif;min-height:100vh;overflow-x:hidden;}}
body::before{{content:'';position:fixed;inset:0;background-image:linear-gradient(rgba(201,160,220,0.04) 1px,transparent 1px),linear-gradient(90deg,rgba(201,160,220,0.04) 1px,transparent 1px);background-size:40px 40px;pointer-events:none;z-index:0;}}
header{{position:relative;z-index:10;padding:16px 28px;border-bottom:1px solid var(--border);display:flex;align-items:center;justify-content:space-between;background:rgba(19,13,26,0.95);backdrop-filter:blur(12px);}}
.logo{{display:flex;align-items:center;gap:12px;}}
.logo-icon{{width:36px;height:36px;background:var(--accent);border-radius:8px;display:flex;align-items:center;justify-content:center;font-size:18px;}}
.logo-text{{font-family:'Cormorant Garamond',serif;font-size:22px;font-weight:700;color:var(--text);letter-spacing:1px;}}
.logo-text span{{color:var(--accent);}}
.header-stats{{display:flex;gap:28px;}}
.hstat{{text-align:center;}}
.hstat-num{{font-family:'Space Mono',monospace;font-size:20px;font-weight:700;color:var(--accent);}}
.hstat-label{{font-size:10px;color:var(--muted);letter-spacing:1px;text-transform:uppercase;}}
.layout{{position:relative;z-index:1;display:grid;grid-template-columns:260px 1fr 280px;height:calc(100vh - 69px);}}
.left-panel{{background:var(--bg2);border-right:1px solid var(--border);overflow-y:auto;padding:20px;}}
.panel-title{{font-size:10px;letter-spacing:2px;text-transform:uppercase;color:var(--accent);margin-bottom:16px;font-family:'Space Mono',monospace;}}
.search-wrap{{position:relative;margin-bottom:20px;}}
.search-wrap input{{width:100%;background:var(--bg3);border:1px solid var(--border);border-radius:8px;padding:10px 14px 10px 36px;color:var(--text);font-family:'Jost',sans-serif;font-size:13px;outline:none;transition:border-color 0.2s;}}
.search-wrap input:focus{{border-color:var(--accent);box-shadow:0 0 0 3px var(--glow);}}
.search-wrap input::placeholder{{color:var(--muted);}}
.search-icon{{position:absolute;left:12px;top:50%;transform:translateY(-50%);color:var(--muted);font-size:14px;}}
.filter-section{{margin-bottom:20px;}}
.filter-label{{font-size:10px;letter-spacing:1.5px;text-transform:uppercase;color:var(--muted);margin-bottom:8px;}}
.filter-chips{{display:flex;flex-wrap:wrap;gap:5px;}}
.chip{{padding:4px 12px;border-radius:20px;font-size:11px;border:1px solid var(--border);background:transparent;color:var(--muted);cursor:pointer;transition:all 0.2s;font-family:'Jost',sans-serif;}}
.chip:hover,.chip.active{{background:var(--accent);border-color:var(--accent);color:#130d1a;font-weight:600;}}
.range-slider{{-webkit-appearance:none;width:100%;height:3px;background:linear-gradient(to right,var(--accent) 0%,var(--accent) 100%,var(--bg3) 100%);border-radius:2px;outline:none;cursor:pointer;}}
.range-slider::-webkit-slider-thumb{{-webkit-appearance:none;width:14px;height:14px;border-radius:50%;background:var(--accent);box-shadow:0 0 8px var(--glow);cursor:pointer;}}
.price-labels{{display:flex;justify-content:space-between;font-size:10px;color:var(--muted);font-family:'Space Mono',monospace;margin-top:6px;}}
.brand-list{{display:flex;flex-direction:column;gap:3px;max-height:200px;overflow-y:auto;}}
.brand-item{{display:flex;align-items:center;gap:8px;padding:5px 6px;border-radius:6px;cursor:pointer;transition:background 0.15s;font-size:11px;}}
.brand-item:hover{{background:var(--bg3);}}
.brand-dot{{width:8px;height:8px;border-radius:50%;flex-shrink:0;}}
.brand-name{{color:var(--text);flex:1;}}
.brand-count{{font-family:'Space Mono',monospace;font-size:10px;color:var(--muted);}}
.brand-item.selected .brand-name{{color:var(--accent);}}
.center-panel{{display:flex;flex-direction:column;overflow:hidden;}}
.viz-toolbar{{padding:10px 16px;border-bottom:1px solid var(--border);display:flex;align-items:center;gap:8px;background:var(--bg2);flex-wrap:wrap;}}
.viz-label{{font-size:11px;color:var(--muted);margin-right:auto;font-family:'Space Mono',monospace;}}
.viz-btn{{padding:5px 12px;border-radius:6px;font-size:11px;border:1px solid var(--border);background:transparent;color:var(--muted);cursor:pointer;font-family:'Jost',sans-serif;transition:all 0.2s;}}
.viz-btn:hover,.viz-btn.active{{border-color:var(--accent);color:var(--accent);}}
canvas#scatterCanvas{{flex:1;display:block;cursor:crosshair;}}
.right-panel{{background:var(--bg2);border-left:1px solid var(--border);overflow-y:auto;display:flex;flex-direction:column;}}
.stats-row{{display:grid;grid-template-columns:1fr 1fr;gap:1px;border-bottom:1px solid var(--border);}}
.stat-card{{padding:14px;background:var(--bg2);}}
.stat-num{{font-family:'Space Mono',monospace;font-size:20px;font-weight:700;color:var(--accent);}}
.stat-num.teal{{color:var(--accent2);}}
.stat-num.pink{{color:var(--accent3);}}
.stat-desc{{font-size:10px;color:var(--muted);margin-top:2px;}}
.detail-area{{flex:1;padding:16px;}}
.no-selection{{height:200px;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:10px;color:var(--muted);text-align:center;}}
.no-selection-icon{{font-size:36px;opacity:0.3;}}
.product-card{{display:none;}}
.product-card.visible{{display:block;}}
.product-brand{{font-size:10px;letter-spacing:2px;text-transform:uppercase;color:var(--accent);font-family:'Space Mono',monospace;margin-bottom:4px;}}
.product-name{{font-size:15px;font-weight:700;line-height:1.3;color:var(--text);margin-bottom:10px;}}
.product-meta{{display:flex;gap:8px;margin-bottom:14px;}}
.meta-pill{{padding:3px 9px;border-radius:20px;font-size:11px;font-family:'Space Mono',monospace;}}
.meta-pill.price{{background:rgba(201,160,220,0.1);color:var(--accent);border:1px solid rgba(201,160,220,0.2);}}
.meta-pill.rank{{background:rgba(232,208,245,0.1);color:var(--accent2);border:1px solid rgba(232,208,245,0.2);}}
.section-title{{font-size:10px;letter-spacing:2px;text-transform:uppercase;color:var(--muted);margin-bottom:8px;}}
.ingredient-tags{{display:flex;flex-wrap:wrap;gap:4px;margin-bottom:14px;}}
.ing-tag{{padding:3px 7px;background:var(--bg3);border:1px solid var(--border);border-radius:4px;font-size:10px;color:var(--muted);font-family:'Space Mono',monospace;}}
.ing-tag.highlight{{background:rgba(201,160,220,0.08);border-color:rgba(201,160,220,0.3);color:var(--accent);}}
.similar-list{{display:flex;flex-direction:column;gap:5px;}}
.similar-item{{padding:8px;background:var(--bg3);border:1px solid var(--border);border-radius:8px;cursor:pointer;transition:all 0.2s;}}
.similar-item:hover{{border-color:var(--accent);}}
.similar-item-brand{{font-size:9px;color:var(--accent);letter-spacing:1px;text-transform:uppercase;}}
.similar-item-name{{font-size:11px;color:var(--text);margin-top:2px;}}
.similar-item-price{{font-size:10px;color:var(--muted);font-family:'Space Mono',monospace;margin-top:3px;}}
#tooltip{{position:fixed;background:var(--bg2);border:1px solid var(--accent);border-radius:8px;padding:8px 12px;pointer-events:none;z-index:1000;display:none;max-width:220px;box-shadow:0 0 20px var(--glow);}}
.tt-brand{{font-size:9px;color:var(--accent);letter-spacing:1.5px;text-transform:uppercase;font-family:'Space Mono',monospace;}}
.tt-name{{font-size:12px;color:var(--text);margin-top:2px;font-weight:600;}}
.tt-price{{font-size:11px;color:var(--accent2);margin-top:3px;font-family:'Space Mono',monospace;}}
::-webkit-scrollbar{{width:3px;}}
::-webkit-scrollbar-track{{background:var(--bg);}}
::-webkit-scrollbar-thumb{{background:var(--border);border-radius:2px;}}
.bottom-bar{{position:fixed;bottom:0;left:0;right:0;height:28px;background:var(--bg2);border-top:1px solid var(--border);display:flex;align-items:center;padding:0 20px;gap:20px;z-index:100;font-family:'Space Mono',monospace;font-size:10px;color:var(--muted);}}
.status-dot{{width:6px;height:6px;border-radius:50%;background:var(--accent);box-shadow:0 0 6px var(--accent);animation:pulse 2s infinite;}}
@keyframes pulse{{0%,100%{{opacity:1;}}50%{{opacity:0.4;}}}}
@media(max-width:768px){{
  .layout{{grid-template-columns:1fr;height:auto;}}
  .left-panel{{border-right:none;border-bottom:1px solid var(--border);max-height:280px;}}
  .right-panel{{border-left:none;border-top:1px solid var(--border);}}
  canvas#scatterCanvas{{height:380px;}}
  .header-stats{{gap:14px;}}
}}
</style>
</head>
<body>

<!-- Loading Screen -->
<div id="loadingScreen" style="position:fixed;inset:0;background:#130d1a;z-index:9999;display:flex;flex-direction:column;align-items:center;justify-content:center;">
  <div style="font-family:'Cormorant Garamond',serif;font-size:48px;font-weight:700;color:#c9a0dc;letter-spacing:4px;margin-bottom:8px;">CosmeticLab</div>
  <div style="font-family:'Space Mono',monospace;font-size:11px;color:#8060a0;letter-spacing:3px;margin-bottom:40px;">INGREDIENT INTELLIGENCE</div>
  <div style="display:flex;gap:8px;">
    <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;animation:pulse 1.2s ease infinite;"></div>
    <div style="width:8px;height:8px;border-radius:50%;background:#e8d0f5;animation:pulse 1.2s ease infinite 0.2s;"></div>
    <div style="width:8px;height:8px;border-radius:50%;background:#f4c2ce;animation:pulse 1.2s ease infinite 0.4s;"></div>
  </div>
</div>

<!-- Sparkle Canvas -->
<canvas id="sparkleCanvas" style="position:fixed;inset:0;pointer-events:none;z-index:0;opacity:0.5;"></canvas>

<header>
  <div class="logo">
    <div class="logo-icon">✦</div>
    <div class="logo-text">Cosmetic<span>Lab</span></div>
  </div>
  <div class="header-stats">
    <div class="hstat"><div class="hstat-num" id="totalCount">190</div><div class="hstat-label">Products</div></div>
    <div class="hstat"><div class="hstat-num" id="brandCount">—</div><div class="hstat-label">Brands</div></div>
    <div class="hstat"><div class="hstat-num" id="visibleCount">190</div><div class="hstat-label">Visible</div></div>
    <div class="hstat"><div class="hstat-num" id="avgPrice">—</div><div class="hstat-label">Avg Price</div></div>
  </div>
</header>

<div class="layout">
  <div class="left-panel">
    <div class="panel-title">// Filters</div>
    <div class="search-wrap">
      <span class="search-icon">&#9906;</span>
      <input type="text" id="searchInput" placeholder="Search products or brands..." oninput="applyFilters()">
    </div>
    <div class="filter-section">
      <div class="filter-label">Skin Type</div>
      <div class="filter-chips">
        <div class="chip active" onclick="toggleSkin(this,'all')">All</div>
        <div class="chip" onclick="toggleSkin(this,'Dry')">Dry</div>
        <div class="chip" onclick="toggleSkin(this,'Normal')">Normal</div>
        <div class="chip" onclick="toggleSkin(this,'Oily')">Oily</div>
        <div class="chip" onclick="toggleSkin(this,'Combination')">Combo</div>
        <div class="chip" onclick="toggleSkin(this,'Sensitive')">Sensitive</div>
      </div>
    </div>
    <div class="filter-section">
      <div class="filter-label">Price Range</div>
      <input type="range" class="range-slider" id="priceRange" min="0" max="500" value="500" step="1" oninput="updatePrice(this)">
      <div class="price-labels"><span>$0</span><span id="priceLabel">Any price</span></div>
    </div>
    <div class="filter-section">
      <div class="filter-label">Color By</div>
      <div class="filter-chips">
        <div class="chip active" onclick="setColorMode(this,'brand')">Brand</div>
        <div class="chip" onclick="setColorMode(this,'price')">Price</div>
        <div class="chip" onclick="setColorMode(this,'rank')">Rank</div>
      </div>
    </div>
    <div class="filter-section">
      <div class="filter-label">Brands</div>
      <div class="brand-list" id="brandList"></div>
    </div>
  </div>

  <div class="center-panel">
    <div class="viz-toolbar">
      <span class="viz-label">t-SNE · Ingredient Similarity Map</span>
      <button class="viz-btn active" onclick="setVizMode(this,'scatter')">Scatter</button>
      <button class="viz-btn" onclick="setVizMode(this,'density')">Density</button>
      <button class="viz-btn" onclick="resetZoom()">Reset View</button>
      <button class="viz-btn" onclick="downloadChart()">Download PNG</button>
      <button class="viz-btn" onclick="shareURL()" id="shareBtn">Share Link</button>
    </div>
    <canvas id="scatterCanvas"></canvas>
  </div>

  <div class="right-panel">
    <div class="stats-row">
      <div class="stat-card"><div class="stat-num" id="statAvgRank">—</div><div class="stat-desc">Avg Rank</div></div>
      <div class="stat-card"><div class="stat-num teal" id="statMinPrice">—</div><div class="stat-desc">Min Price</div></div>
      <div class="stat-card"><div class="stat-num pink" id="statMaxPrice">—</div><div class="stat-desc">Max Price</div></div>
      <div class="stat-card"><div class="stat-num" id="statIngCount">—</div><div class="stat-desc">Avg Ingredients</div></div>
    </div>
    <div class="detail-area">
      <div class="no-selection" id="noSelection">
        <div class="no-selection-icon">✦</div>
        <div style="font-size:12px;color:var(--muted);text-align:center;line-height:1.6;">Click any dot on the map<br>to explore product details</div>
      </div>
      <div class="product-card" id="productCard">
        <div class="product-brand" id="pdBrand"></div>
        <div class="product-name" id="pdName"></div>
        <div class="product-meta">
          <div class="meta-pill price" id="pdPrice"></div>
          <div class="meta-pill rank" id="pdRank"></div>
        </div>
        <div class="section-title">// Ingredients</div>
        <div class="ingredient-tags" id="pdIngredients"></div>
        <div class="section-title">// Similar Products</div>
        <div class="similar-list" id="similarList"></div>
      </div>
    </div>
  </div>
</div>

<div id="tooltip">
  <div class="tt-brand" id="ttBrand"></div>
  <div class="tt-name" id="ttName"></div>
  <div class="tt-price" id="ttPrice"></div>
</div>

<div class="bottom-bar">
  <div class="status-dot"></div>
  <span style="color:var(--accent);">COSMETICLAB</span>
  <span>Sephora · 190 Moisturizers · Dry Skin</span>
  <span style="margin-left:auto">Visible: <span style="color:var(--accent);" id="footerVisible">190</span></span>
</div>

<script>
const PRODUCTS = {data_json};
const BRAND_COLORS = ['#c9a0dc','#e8d0f5','#f4c2ce','#9b7fd4','#d4a0c8','#b888d0','#e8b8d8','#a870c0','#f0c8e8','#c0a0e8','#d8b0f0','#e0c0d8','#b898c8','#f8d0e8','#c8b0e0','#e0a8d0','#d0b8e8','#b8a0d8','#e8c0e0','#c8a8d8','#d8c0f0','#b8b0d8','#e8d0e8'];
let state={{search:'',skinFilter:'all',maxPrice:500,colorMode:'brand',selectedBrands:new Set(),vizMode:'scatter',selectedIdx:-1,zoom:1,panX:0,panY:0,dragging:false,dragStart:null}};
let filteredIndices=[],brandColorMap={{}},canvas,ctx,hoveredIdx=-1;

window.addEventListener('load',()=>{{
  setTimeout(()=>{{
    const ls=document.getElementById('loadingScreen');
    ls.style.transition='opacity 0.8s';
    ls.style.opacity='0';
    setTimeout(()=>ls.style.display='none',800);
  }},2200);

  canvas=document.getElementById('scatterCanvas');
  ctx=canvas.getContext('2d');
  resizeCanvas();
  window.addEventListener('resize',resizeCanvas);
  canvas.addEventListener('mousemove',onMouseMove);
  canvas.addEventListener('click',onCanvasClick);
  canvas.addEventListener('wheel',onWheel,{{passive:false}});
  canvas.addEventListener('mousedown',e=>{{state.dragging=true;state.dragStart={{x:e.clientX-state.panX,y:e.clientY-state.panY}};}});
  canvas.addEventListener('mouseup',()=>state.dragging=false);
  canvas.addEventListener('mouseleave',()=>{{state.dragging=false;document.getElementById('tooltip').style.display='none';}});
  initSparkles();
  buildBrandList();
  updateHeaderStats();
  applyFilters();
}});

function initSparkles(){{
  const sc=document.getElementById('sparkleCanvas');
  const sctx=sc.getContext('2d');
  sc.width=window.innerWidth;sc.height=window.innerHeight;
  window.addEventListener('resize',()=>{{sc.width=window.innerWidth;sc.height=window.innerHeight;}});
  const particles=Array.from({{length:50}},()=>{{
    const colors=['#c9a0dc','#e8d0f5','#f4c2ce','#9b7fd4'];
    return{{x:Math.random()*sc.width,y:Math.random()*sc.height,r:Math.random()*1.5+0.3,dx:(Math.random()-0.5)*0.2,dy:(Math.random()-0.5)*0.2,op:Math.random(),dop:(Math.random()-0.5)*0.008,color:colors[Math.floor(Math.random()*4)]}};
  }});
  function animSpark(){{
    sctx.clearRect(0,0,sc.width,sc.height);
    particles.forEach(p=>{{
      p.x+=p.dx;p.y+=p.dy;p.op+=p.dop;
      if(p.op<=0||p.op>=1)p.dop*=-1;
      if(p.x<0||p.x>sc.width)p.dx*=-1;
      if(p.y<0||p.y>sc.height)p.dy*=-1;
      sctx.beginPath();sctx.arc(p.x,p.y,p.r,0,Math.PI*2);
      sctx.fillStyle=p.color;sctx.globalAlpha=p.op;sctx.fill();
    }});
    sctx.globalAlpha=1;
    requestAnimationFrame(animSpark);
  }}
  animSpark();
}}

function buildBrandList(){{
  const counts={{}};
  PRODUCTS.forEach(p=>{{counts[p.Brand]=(counts[p.Brand]||0)+1;}});
  const bl=Object.entries(counts).sort((a,b)=>b[1]-a[1]);
  bl.forEach(([b],i)=>{{brandColorMap[b]=BRAND_COLORS[i%BRAND_COLORS.length];}});
  document.getElementById('brandList').innerHTML=bl.map(([b,c],i)=>`
    <div class="brand-item" onclick="toggleBrand('${{b.replace(/'/g,"\\'")}},this)" data-brand="${{b}}">
      <div class="brand-dot" style="background:${{BRAND_COLORS[i%BRAND_COLORS.length]}}"></div>
      <span class="brand-name">${{b}}</span>
      <span class="brand-count">${{c}}</span>
    </div>`).join('');
  document.getElementById('brandCount').textContent=bl.length;
}}

function updateHeaderStats(){{
  const prices=PRODUCTS.map(p=>p.Price);
  document.getElementById('avgPrice').textContent='$'+Math.round(prices.reduce((a,b)=>a+b,0)/prices.length);
  document.getElementById('statAvgRank').textContent=(PRODUCTS.map(p=>p.Rank).reduce((a,b)=>a+b,0)/PRODUCTS.length).toFixed(1)+'*';
  document.getElementById('statMinPrice').textContent='$'+Math.min(...prices);
  document.getElementById('statMaxPrice').textContent='$'+Math.max(...prices);
  const ings=PRODUCTS.map(p=>p.Ingredients.split(',').length);
  document.getElementById('statIngCount').textContent=Math.round(ings.reduce((a,b)=>a+b,0)/ings.length);
}}

function applyFilters(){{
  state.search=document.getElementById('searchInput').value.toLowerCase();
  filteredIndices=PRODUCTS.map((_,i)=>i).filter(i=>{{
    const p=PRODUCTS[i];
    if(state.search&&!p.Name.toLowerCase().includes(state.search)&&!p.Brand.toLowerCase().includes(state.search))return false;
    if(state.skinFilter!=='all'&&!p[state.skinFilter])return false;
    if(p.Price>state.maxPrice)return false;
    if(state.selectedBrands.size>0&&!state.selectedBrands.has(p.Brand))return false;
    return true;
  }});
  document.getElementById('visibleCount').textContent=filteredIndices.length;
  document.getElementById('footerVisible').textContent=filteredIndices.length;
  drawCanvas();
}}

function toggleSkin(el,val){{
  document.querySelectorAll('.filter-chips .chip').forEach(c=>{{if(c.onclick&&c.onclick.toString().includes('toggleSkin'))c.classList.remove('active');}});
  el.classList.add('active');state.skinFilter=val;applyFilters();
}}
function updatePrice(el){{
  state.maxPrice=parseInt(el.value);
  const pct=state.maxPrice/500*100;
  el.style.background=`linear-gradient(to right,var(--accent) 0%,var(--accent) ${{pct}}%,var(--bg3) ${{pct}}%)`;
  document.getElementById('priceLabel').textContent=state.maxPrice>=500?'Any price':'Up to $'+state.maxPrice;
  applyFilters();
}}
function toggleBrand(brand,el){{
  if(!el)return;
  el.classList.toggle('selected');
  if(state.selectedBrands.has(brand))state.selectedBrands.delete(brand);
  else state.selectedBrands.add(brand);
  applyFilters();
}}
function setColorMode(el,mode){{
  document.querySelectorAll('.filter-chips .chip').forEach(c=>{{if(c.onclick&&c.onclick.toString().includes('setColorMode'))c.classList.remove('active');}});
  el.classList.add('active');state.colorMode=mode;drawCanvas();
}}
function setVizMode(el,mode){{
  document.querySelectorAll('.viz-btn').forEach(b=>b.classList.remove('active'));
  el.classList.add('active');state.vizMode=mode;drawCanvas();
}}
function resetZoom(){{state.zoom=1;state.panX=0;state.panY=0;drawCanvas();}}
function resizeCanvas(){{
  const p=canvas.parentElement;
  canvas.width=p.clientWidth;
  canvas.height=p.clientHeight-41;
  drawCanvas();
}}
function getColor(p){{
  if(state.colorMode==='brand')return brandColorMap[p.Brand]||'#c9a0dc';
  if(state.colorMode==='price'){{
    const prices=PRODUCTS.map(q=>q.Price);
    const t=(p.Price-Math.min(...prices))/(Math.max(...prices)-Math.min(...prices));
    return`rgb(${{Math.round(201+54*t)}},${{Math.round(160-94*t)}},${{Math.round(220-56*t)}})`;
  }}
  if(state.colorMode==='rank'){{
    const t=(p.Rank-3)/2;
    return`rgb(${{Math.round(244*t+201*(1-t))}},${{Math.round(194*(1-t)+160*t)}},${{Math.round(206*(1-t)+220*t)}})`;
  }}
  return'#c9a0dc';
}}
function worldToScreen(x,y){{
  const cx=canvas.width/2+state.panX,cy=canvas.height/2+state.panY;
  const sc=(Math.min(canvas.width,canvas.height)/900)*state.zoom;
  return{{x:cx+x*sc,y:cy+y*sc}};
}}
function drawCanvas(){{
  if(!ctx)return;
  ctx.clearRect(0,0,canvas.width,canvas.height);
  const fs=new Set(filteredIndices);
  PRODUCTS.forEach((p,i)=>{{
    if(fs.has(i))return;
    const{{x,y}}=worldToScreen(p.X,p.Y);
    ctx.beginPath();ctx.arc(x,y,3,0,Math.PI*2);
    ctx.fillStyle='rgba(150,100,180,0.15)';ctx.fill();
  }});
  if(state.vizMode==='density'){{
    filteredIndices.forEach(i=>{{
      const p=PRODUCTS[i],{{x,y}}=worldToScreen(p.X,p.Y),color=getColor(p);
      const g=ctx.createRadialGradient(x,y,0,x,y,28);
      g.addColorStop(0,color+'40');g.addColorStop(1,color+'00');
      ctx.beginPath();ctx.arc(x,y,28,0,Math.PI*2);ctx.fillStyle=g;ctx.fill();
    }});
  }}
  filteredIndices.forEach(i=>{{
    const p=PRODUCTS[i],{{x,y}}=worldToScreen(p.X,p.Y),color=getColor(p),isSel=state.selectedIdx===i,r=isSel?10:7;
    if(isSel){{
      const g=ctx.createRadialGradient(x,y,0,x,y,22);
      g.addColorStop(0,color+'60');g.addColorStop(1,color+'00');
      ctx.beginPath();ctx.arc(x,y,22,0,Math.PI*2);ctx.fillStyle=g;ctx.fill();
    }}
    ctx.beginPath();ctx.arc(x,y,r,0,Math.PI*2);
    ctx.fillStyle=isSel?color:color+'dd';ctx.fill();
    if(isSel){{ctx.beginPath();ctx.arc(x,y,r+3,0,Math.PI*2);ctx.strokeStyle=color;ctx.lineWidth=1.5;ctx.stroke();}}
  }});
}}
function onMouseMove(e){{
  if(state.dragging&&state.dragStart){{
    state.panX=e.clientX-state.dragStart.x;state.panY=e.clientY-state.dragStart.y;drawCanvas();return;
  }}
  const rect=canvas.getBoundingClientRect(),mx=e.clientX-rect.left,my=e.clientY-rect.top;
  let closest=-1,cd=18;
  filteredIndices.forEach(i=>{{
    const p=PRODUCTS[i],{{x,y}}=worldToScreen(p.X,p.Y),d=Math.hypot(mx-x,my-y);
    if(d<cd){{closest=i;cd=d;}}
  }});
  const tt=document.getElementById('tooltip');
  if(closest>=0){{
    canvas.style.cursor='pointer';
    const p=PRODUCTS[closest];
    document.getElementById('ttBrand').textContent=p.Brand;
    document.getElementById('ttName').textContent=p.Name;
    document.getElementById('ttPrice').textContent='$'+p.Price+' · '+p.Rank+' stars';
    tt.style.display='block';tt.style.left=(e.clientX+14)+'px';tt.style.top=(e.clientY-10)+'px';
    hoveredIdx=closest;
  }}else{{canvas.style.cursor='crosshair';tt.style.display='none';hoveredIdx=-1;}}
}}
function onCanvasClick(){{
  if(hoveredIdx>=0){{state.selectedIdx=hoveredIdx;showDetail(PRODUCTS[hoveredIdx],hoveredIdx);drawCanvas();}}
}}
function onWheel(e){{
  e.preventDefault();state.zoom=Math.max(0.3,Math.min(5,state.zoom*(e.deltaY>0?0.9:1.1)));drawCanvas();
}}
function showDetail(p,idx){{
  document.getElementById('noSelection').style.display='none';
  const card=document.getElementById('productCard');card.classList.add('visible');
  document.getElementById('pdBrand').textContent=p.Brand;
  document.getElementById('pdName').textContent=p.Name;
  document.getElementById('pdPrice').textContent='$'+p.Price;
  document.getElementById('pdRank').textContent=p.Rank+' stars';
  const ings=p.Ingredients.split(',').map(s=>s.trim()).filter(Boolean);
  const keys=['Water','Glycerin','Niacinamide','Hyaluronic Acid','Ceramide','Retinol','Vitamin C','Peptides','Squalane'];
  document.getElementById('pdIngredients').innerHTML=ings.slice(0,18).map(ing=>`<span class="ing-tag ${{keys.some(k=>ing.toLowerCase().includes(k.toLowerCase()))?'highlight':''}}">${{ing}}</span>`).join('');
  const dists=PRODUCTS.map((q,i)=>{{return{{i,d:Math.hypot(q.X-p.X,q.Y-p.Y)}}}}).filter(d=>d.i!==idx).sort((a,b)=>a.d-b.d).slice(0,4);
  document.getElementById('similarList').innerHTML=dists.map(({{i}})=>{{
    const q=PRODUCTS[i];
    return`<div class="similar-item" onclick="selectProduct(${{i}})"><div class="similar-item-brand">${{q.Brand}}</div><div class="similar-item-name">${{q.Name}}</div><div class="similar-item-price">$${{q.Price}} · ${{q.Rank}} stars</div></div>`;
  }}).join('');
}}
function selectProduct(idx){{state.selectedIdx=idx;showDetail(PRODUCTS[idx],idx);drawCanvas();}}
function downloadChart(){{
  const link=document.createElement('a');
  link.download='cosmetic_map.png';
  link.href=canvas.toDataURL('image/png');
  link.click();
}}
function shareURL(){{
  const url='https://storage.googleapis.com/cosmetic-map-aishu/cosmetic_dashboard.html';
  navigator.clipboard.writeText(url).then(()=>{{
    const btn=document.getElementById('shareBtn');
    btn.textContent='Copied!';btn.style.color='#c9a0dc';
    setTimeout(()=>{{btn.textContent='Share Link';btn.style.color='';}},2000);
  }});
}}
</script>

</body>
</html>'''

with open('cosmetic_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(dashboard)

print("✅ Clean dashboard built!")
print("Now open cosmetic_dashboard.html in your browser!")

✅ Clean dashboard built!
Now open cosmetic_dashboard.html in your browser!


In [71]:
with open('cosmetic_dashboard.html', 'r') as f:
    html = f.read()

if 'ANALYTICS OVERVIEW' in html:
    print("✅ Charts already there - just scroll down!")
elif 'Chart.js' in html:
    print("✅ Charts already there - just scroll down!")
else:
    print("❌ Charts missing - need to re-add them")

❌ Charts missing - need to re-add them


In [72]:
with open('cosmetic_dashboard.html', 'r') as f:
    html = f.read()

from collections import Counter
import json

# Rebuild all chart data
brand_counts = moisturizers_dry['Brand'].value_counts().head(10)
brand_data = [{"brand": b, "count": int(c)} for b, c in brand_counts.items()]

bins = [0, 30, 60, 100, 200, 500]
labels = ['$0-30', '$30-60', '$60-100', '$100-200', '$200+']
price_data = [{"label": labels[i], "count": int(((moisturizers_dry['Price'] >= bins[i]) & (moisturizers_dry['Price'] < bins[i+1])).sum())} for i in range(len(bins)-1)]

rank_counts = moisturizers_dry['Rank'].value_counts().sort_index()
rank_data = [{"rank": str(round(float(r),1)), "count": int(c)} for r, c in rank_counts.items()]

all_ingredients = []
for ing_list in moisturizers_dry['Ingredients']:
    for ing in ing_list.split(','):
        ing = ing.strip().lower()
        if len(ing) > 2:
            all_ingredients.append(ing)
top_ingredients = Counter(all_ingredients).most_common(15)
ing_data = [{"name": k.title(), "count": v} for k, v in top_ingredients]

# Level 2 data
key_ingredients = {
    'Hyaluronic Acid': 'Hydration','Glycerin': 'Hydration',
    'Ceramide': 'Barrier','Niacinamide': 'Brightening',
    'Retinol': 'Anti-aging','Vitamin C': 'Brightening',
    'Peptides': 'Anti-aging','Squalane': 'Barrier',
    'Aloe': 'Soothing','Zinc': 'Oil Control',
    'Salicylic Acid': 'Oil Control','Collagen': 'Anti-aging',
    'Vitamin E': 'Barrier','Caffeine': 'Depuffing'
}
scores = []
for _, row in moisturizers_dry.iterrows():
    ing_lower = row['Ingredients'].lower()
    found = [{'ingredient': ing, 'benefit': benefit} for ing, benefit in key_ingredients.items() if ing.lower() in ing_lower]
    scores.append({'Name': row['Name'], 'Brand': row['Brand'], 'Price': float(row['Price']), 'Rank': float(row['Rank']), 'score': len(found), 'key_ingredients': found})

top_products = sorted(scores, key=lambda x: x['score'], reverse=True)[:8]
benefit_counts = Counter()
for s in scores:
    for ki in s['key_ingredients']:
        benefit_counts[ki['benefit']] += 1

brands_top = moisturizers_dry['Brand'].value_counts().head(8).index.tolist()
heatmap_data = []
for b1 in brands_top:
    row_data = []
    ings1 = set()
    for _, r in moisturizers_dry[moisturizers_dry['Brand']==b1].iterrows():
        ings1.update([i.strip().lower() for i in r['Ingredients'].split(',')])
    for b2 in brands_top:
        ings2 = set()
        for _, r in moisturizers_dry[moisturizers_dry['Brand']==b2].iterrows():
            ings2.update([i.strip().lower() for i in r['Ingredients'].split(',')])
        overlap = len(ings1 & ings2) / max(len(ings1 | ings2), 1)
        row_data.append(round(overlap * 100, 1))
    heatmap_data.append(row_data)

# Level 3 data
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
similarity_matrix = cosine_similarity(A)
recommendations = []
for i in range(len(moisturizers_dry)):
    sim_scores = sorted(enumerate(similarity_matrix[i]), key=lambda x: x[1], reverse=True)
    sim_scores = [s for s in sim_scores if s[0] != i][:5]
    recommendations.append([{'Name': moisturizers_dry.iloc[idx]['Name'], 'Brand': moisturizers_dry.iloc[idx]['Brand'], 'Price': float(moisturizers_dry.iloc[idx]['Price']), 'Rank': float(moisturizers_dry.iloc[idx]['Rank']), 'similarity': round(float(score)*100,1)} for idx, score in sim_scores])

concern_ingredients = {
    'dryness': ['glycerin','hyaluronic acid','squalane','ceramide','shea butter'],
    'aging': ['retinol','peptide','collagen','niacinamide','vitamin c'],
    'oily': ['salicylic acid','zinc','niacinamide','clay'],
    'sensitive': ['aloe','ceramide','oat','centella','allantoin'],
    'brightening': ['vitamin c','niacinamide','kojic','arbutin','glycolic'],
    'pores': ['salicylic acid','niacinamide','retinol','clay','zinc']
}

charts_json = json.dumps({"brands": brand_data, "prices": price_data, "ranks": rank_data, "ingredients": ing_data})
level2_json = json.dumps({"top_products": top_products, "benefit_counts": dict(benefit_counts), "brands": brands_top, "heatmap": heatmap_data})
rec_json = json.dumps({"recommendations": recommendations, "concern_ingredients": concern_ingredients})

all_sections = f'''
<div style="position:relative;z-index:1;background:#1e1228;border-top:1px solid rgba(201,160,220,0.15);padding:24px 32px 40px;">
  <div style="display:flex;align-items:center;gap:12px;margin-bottom:24px;">
    <span style="font-family:'Space Mono',monospace;font-size:10px;letter-spacing:2px;color:#c9a0dc;">// ANALYTICS OVERVIEW</span>
    <div style="flex:1;height:1px;background:rgba(201,160,220,0.1);"></div>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:16px;">
    <div style="background:#261530;border:1px solid rgba(201,160,220,0.15);border-radius:12px;padding:16px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#8060a0;letter-spacing:1px;margin-bottom:12px;">TOP 10 BRANDS</div>
      <canvas id="brandChart" height="220"></canvas>
    </div>
    <div style="background:#261530;border:1px solid rgba(201,160,220,0.15);border-radius:12px;padding:16px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#8060a0;letter-spacing:1px;margin-bottom:12px;">PRICE DISTRIBUTION</div>
      <canvas id="priceChart" height="220"></canvas>
    </div>
    <div style="background:#261530;border:1px solid rgba(201,160,220,0.15);border-radius:12px;padding:16px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#8060a0;letter-spacing:1px;margin-bottom:12px;">RATING DISTRIBUTION</div>
      <canvas id="rankChart" height="220"></canvas>
    </div>
    <div style="background:#261530;border:1px solid rgba(201,160,220,0.15);border-radius:12px;padding:16px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#8060a0;letter-spacing:1px;margin-bottom:12px;">TOP 15 INGREDIENTS</div>
      <canvas id="ingChart" height="220"></canvas>
    </div>
  </div>
</div>

<div style="position:relative;z-index:1;background:#130d1a;border-top:1px solid rgba(201,160,220,0.15);padding:24px 32px 40px;">
  <div style="display:flex;align-items:center;gap:12px;margin-bottom:24px;">
    <span style="font-family:'Space Mono',monospace;font-size:10px;letter-spacing:2px;color:#e8d0f5;">// INGREDIENT INTELLIGENCE</span>
    <div style="flex:1;height:1px;background:rgba(232,208,245,0.1);"></div>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;">
    <div style="background:#1e1228;border:1px solid rgba(201,160,220,0.15);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#8060a0;letter-spacing:1px;margin-bottom:14px;">TOP PRODUCTS BY BENEFICIAL INGREDIENTS</div>
      <div id="topProductsList"></div>
    </div>
    <div style="background:#1e1228;border:1px solid rgba(201,160,220,0.15);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#8060a0;letter-spacing:1px;margin-bottom:14px;">BRAND INGREDIENT OVERLAP HEATMAP</div>
      <div id="heatmapContainer"></div>
    </div>
    <div style="background:#1e1228;border:1px solid rgba(201,160,220,0.15);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#8060a0;letter-spacing:1px;margin-bottom:14px;">BENEFIT CATEGORY BREAKDOWN</div>
      <canvas id="benefitChart" height="200"></canvas>
    </div>
    <div style="background:#1e1228;border:1px solid rgba(201,160,220,0.15);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#8060a0;letter-spacing:1px;margin-bottom:12px;">FIND PRODUCTS BY INGREDIENT</div>
      <input type="text" id="ingSearch" placeholder="Type e.g. Retinol, Ceramide..."
        style="width:100%;background:#130d1a;border:1px solid rgba(201,160,220,0.2);border-radius:8px;padding:10px;color:#f0e0ff;font-family:'Space Mono',monospace;font-size:11px;outline:none;margin-bottom:10px;">
      <div id="ingResults" style="display:flex;flex-direction:column;gap:5px;max-height:200px;overflow-y:auto;"></div>
    </div>
  </div>
</div>

<div style="position:relative;z-index:1;background:#1e1228;border-top:1px solid rgba(244,194,206,0.2);padding:24px 32px 40px;">
  <div style="display:flex;align-items:center;gap:12px;margin-bottom:24px;">
    <span style="font-family:'Space Mono',monospace;font-size:10px;letter-spacing:2px;color:#f4c2ce;">// RECOMMENDATION ENGINE</span>
    <div style="flex:1;height:1px;background:rgba(244,194,206,0.1);"></div>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:16px;">
    <div style="background:#261530;border:1px solid rgba(244,194,206,0.15);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#8060a0;letter-spacing:1px;margin-bottom:14px;">GET PERSONALIZED RECOMMENDATIONS</div>
      <div style="font-size:11px;color:#8060a0;margin-bottom:8px;">YOUR SKIN CONCERNS</div>
      <div style="display:flex;flex-wrap:wrap;gap:6px;margin-bottom:14px;" id="concernChips">
        <div onclick="toggleConcern(this,'dryness')" style="padding:6px 14px;border-radius:20px;font-size:11px;border:1px solid rgba(244,194,206,0.3);color:#8060a0;cursor:pointer;transition:all 0.2s;font-family:'Space Mono',monospace;">Dryness</div>
        <div onclick="toggleConcern(this,'aging')" style="padding:6px 14px;border-radius:20px;font-size:11px;border:1px solid rgba(244,194,206,0.3);color:#8060a0;cursor:pointer;transition:all 0.2s;font-family:'Space Mono',monospace;">Aging</div>
        <div onclick="toggleConcern(this,'oily')" style="padding:6px 14px;border-radius:20px;font-size:11px;border:1px solid rgba(244,194,206,0.3);color:#8060a0;cursor:pointer;transition:all 0.2s;font-family:'Space Mono',monospace;">Oily Skin</div>
        <div onclick="toggleConcern(this,'sensitive')" style="padding:6px 14px;border-radius:20px;font-size:11px;border:1px solid rgba(244,194,206,0.3);color:#8060a0;cursor:pointer;transition:all 0.2s;font-family:'Space Mono',monospace;">Sensitive</div>
        <div onclick="toggleConcern(this,'brightening')" style="padding:6px 14px;border-radius:20px;font-size:11px;border:1px solid rgba(244,194,206,0.3);color:#8060a0;cursor:pointer;transition:all 0.2s;font-family:'Space Mono',monospace;">Brightening</div>
        <div onclick="toggleConcern(this,'pores')" style="padding:6px 14px;border-radius:20px;font-size:11px;border:1px solid rgba(244,194,206,0.3);color:#8060a0;cursor:pointer;transition:all 0.2s;font-family:'Space Mono',monospace;">Pores</div>
      </div>
      <div style="font-size:11px;color:#8060a0;margin-bottom:6px;">MAX BUDGET</div>
      <div style="display:flex;align-items:center;gap:10px;margin-bottom:14px;">
        <input type="range" id="budgetSlider" min="10" max="300" value="100" step="10"
          style="flex:1;-webkit-appearance:none;height:3px;background:linear-gradient(to right,#c9a0dc 0%,#c9a0dc 33%,#261530 33%);border-radius:2px;outline:none;cursor:pointer;"
          oninput="updateBudget(this)">
        <span id="budgetLabel" style="font-family:'Space Mono',monospace;font-size:13px;color:#c9a0dc;min-width:50px;">$100</span>
      </div>
      <button onclick="getRecommendations()" style="width:100%;padding:12px;background:linear-gradient(135deg,#c9a0dc,#9b7fd4);border:none;border-radius:8px;color:#fff;font-family:'Space Mono',monospace;font-size:12px;font-weight:700;cursor:pointer;letter-spacing:1px;">
        GET MY RECOMMENDATIONS
      </button>
      <div id="recResults" style="margin-top:14px;display:flex;flex-direction:column;gap:7px;"></div>
    </div>
    <div style="background:#261530;border:1px solid rgba(244,194,206,0.15);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#8060a0;letter-spacing:1px;margin-bottom:14px;">FIND SIMILAR PRODUCTS (DUPES FINDER)</div>
      <input type="text" id="dupeSearch" placeholder="Type product name..."
        style="width:100%;background:#130d1a;border:1px solid rgba(201,160,220,0.2);border-radius:8px;padding:10px;color:#f0e0ff;font-family:'Space Mono',monospace;font-size:11px;outline:none;margin-bottom:8px;"
        oninput="searchProducts(this.value)">
      <div id="productDropdown" style="display:flex;flex-direction:column;gap:4px;max-height:140px;overflow-y:auto;margin-bottom:10px;"></div>
      <div id="dupeResults">
        <div style="text-align:center;padding:24px;color:#8060a0;font-family:'Space Mono',monospace;font-size:11px;">Search for a product to find its dupes</div>
      </div>
    </div>
  </div>
</div>

<div style="position:relative;z-index:1;background:#130d1a;border-top:1px solid rgba(201,160,220,0.15);padding:32px 32px 60px;">
  <div style="display:flex;align-items:center;gap:12px;margin-bottom:28px;">
    <span style="font-family:'Space Mono',monospace;font-size:10px;letter-spacing:2px;color:#c9a0dc;">// ABOUT THIS PROJECT</span>
    <div style="flex:1;height:1px;background:rgba(201,160,220,0.1);"></div>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:16px;margin-bottom:28px;">
    <div style="background:#1e1228;border:1px solid rgba(201,160,220,0.15);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#c9a0dc;letter-spacing:1px;margin-bottom:10px;">BUILT BY</div>
      <div style="font-family:'Cormorant Garamond',serif;font-size:24px;font-weight:700;color:#f0e0ff;margin-bottom:4px;">Aiswarya</div>
      <div style="font-size:12px;color:#8060a0;font-family:'Space Mono',monospace;margin-bottom:14px;">Data Analytics Portfolio Project</div>
      <div style="display:flex;flex-direction:column;gap:5px;">
        <div style="font-size:11px;color:#8060a0;">&#8594; Machine Learning · t-SNE</div>
        <div style="font-size:11px;color:#8060a0;">&#8594; NLP · Ingredient Tokenization</div>
        <div style="font-size:11px;color:#8060a0;">&#8594; Cosine Similarity · Recommendations</div>
        <div style="font-size:11px;color:#8060a0;">&#8594; Data Visualization · Analytics</div>
        <div style="font-size:11px;color:#8060a0;">&#8594; Google Cloud · Deployment</div>
      </div>
    </div>
    <div style="background:#1e1228;border:1px solid rgba(201,160,220,0.15);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#e8d0f5;letter-spacing:1px;margin-bottom:12px;">TECH STACK</div>
      <div style="display:flex;flex-direction:column;gap:8px;">
        <div style="display:flex;justify-content:space-between;align-items:center;">
          <span style="font-size:12px;color:#f0e0ff;">Python</span>
          <div style="display:flex;gap:3px;">
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
          </div>
        </div>
        <div style="display:flex;justify-content:space-between;align-items:center;">
          <span style="font-size:12px;color:#f0e0ff;">Scikit-learn</span>
          <div style="display:flex;gap:3px;">
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#8060a0;"></div>
          </div>
        </div>
        <div style="display:flex;justify-content:space-between;align-items:center;">
          <span style="font-size:12px;color:#f0e0ff;">Chart.js</span>
          <div style="display:flex;gap:3px;">
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#8060a0;"></div>
          </div>
        </div>
        <div style="display:flex;justify-content:space-between;align-items:center;">
          <span style="font-size:12px;color:#f0e0ff;">Google Cloud</span>
          <div style="display:flex;gap:3px;">
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#8060a0;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#8060a0;"></div>
          </div>
        </div>
        <div style="display:flex;justify-content:space-between;align-items:center;">
          <span style="font-size:12px;color:#f0e0ff;">HTML/CSS/JS</span>
          <div style="display:flex;gap:3px;">
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
            <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;"></div>
          </div>
        </div>
      </div>
    </div>
    <div style="background:#1e1228;border:1px solid rgba(201,160,220,0.15);border-radius:12px;padding:20px;">
      <div style="font-family:'Space Mono',monospace;font-size:10px;color:#f4c2ce;letter-spacing:1px;margin-bottom:12px;">PROJECT STATS</div>
      <div style="display:flex;flex-direction:column;gap:10px;">
        <div style="display:flex;justify-content:space-between;padding-bottom:8px;border-bottom:1px solid rgba(201,160,220,0.08);">
          <span style="font-size:11px;color:#8060a0;">Dataset</span>
          <span style="font-family:'Space Mono',monospace;font-size:11px;color:#c9a0dc;">1,472 products</span>
        </div>
        <div style="display:flex;justify-content:space-between;padding-bottom:8px;border-bottom:1px solid rgba(201,160,220,0.08);">
          <span style="font-size:11px;color:#8060a0;">Analyzed</span>
          <span style="font-family:'Space Mono',monospace;font-size:11px;color:#c9a0dc;">190 moisturizers</span>
        </div>
        <div style="display:flex;justify-content:space-between;padding-bottom:8px;border-bottom:1px solid rgba(201,160,220,0.08);">
          <span style="font-size:11px;color:#8060a0;">Unique ingredients</span>
          <span style="font-family:'Space Mono',monospace;font-size:11px;color:#e8d0f5;">2,233</span>
        </div>
        <div style="display:flex;justify-content:space-between;padding-bottom:8px;border-bottom:1px solid rgba(201,160,220,0.08);">
          <span style="font-size:11px;color:#8060a0;">Brands covered</span>
          <span style="font-family:'Space Mono',monospace;font-size:11px;color:#e8d0f5;">53 brands</span>
        </div>
        <div style="display:flex;justify-content:space-between;padding-bottom:8px;border-bottom:1px solid rgba(201,160,220,0.08);">
          <span style="font-size:11px;color:#8060a0;">ML Algorithm</span>
          <span style="font-family:'Space Mono',monospace;font-size:11px;color:#f4c2ce;">t-SNE + Cosine</span>
        </div>
        <div style="display:flex;justify-content:space-between;">
          <span style="font-size:11px;color:#8060a0;">Deployed on</span>
          <span style="font-family:'Space Mono',monospace;font-size:11px;color:#f4c2ce;">Google Cloud</span>
        </div>
      </div>
    </div>
  </div>
  <div style="text-align:center;padding-top:20px;border-top:1px solid rgba(201,160,220,0.08);">
    <div style="font-family:'Space Mono',monospace;font-size:11px;color:#8060a0;">
      Built with love by <span style="color:#c9a0dc;">Aiswarya</span> &nbsp;·&nbsp;
      <a href="https://www.linkedin.com/in/aiswarya-" target="_blank" style="color:#e8d0f5;text-decoration:none;">LinkedIn</a>
      &nbsp;·&nbsp;
      <a href="https://github.com/Aiswarya0109" target="_blank" style="color:#f4c2ce;text-decoration:none;">GitHub</a>
      &nbsp;·&nbsp;
      Data from <span style="color:#e8d0f5;">Sephora via Kaggle</span>
    </div>
  </div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<script>
const CHART_DATA = {charts_json};
const L2_DATA = {level2_json};
const REC_DATA = {rec_json};
let selectedConcerns = new Set();
let maxBudget = 100;

Chart.defaults.color = '#8060a0';
Chart.defaults.borderColor = 'rgba(201,160,220,0.08)';
Chart.defaults.font.family = "'Space Mono', monospace";
Chart.defaults.font.size = 10;

new Chart(document.getElementById('brandChart'), {{
  type: 'bar',
  data: {{
    labels: CHART_DATA.brands.map(d => d.brand),
    datasets: [{{ data: CHART_DATA.brands.map(d => d.count),
      backgroundColor: CHART_DATA.brands.map((_,i) => `hsl(${{280+i*12}},50%,${{60+i*2}}%)`),
      borderRadius: 4, borderSkipped: false }}]
  }},
  options: {{ indexAxis: 'y', responsive: true,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ grid: {{ color: 'rgba(201,160,220,0.06)' }}, ticks: {{ color: '#8060a0' }} }},
      y: {{ grid: {{ display: false }}, ticks: {{ color: '#c9a0dc', font: {{ size: 9 }} }} }}
    }}
  }}
}});

new Chart(document.getElementById('priceChart'), {{
  type: 'doughnut',
  data: {{
    labels: CHART_DATA.prices.map(d => d.label),
    datasets: [{{ data: CHART_DATA.prices.map(d => d.count),
      backgroundColor: ['#c9a0dc','#e8d0f5','#f4c2ce','#9b7fd4','#d4a0c8'],
      borderColor: '#1e1228', borderWidth: 3 }}]
  }},
  options: {{ responsive: true,
    plugins: {{ legend: {{ position: 'bottom', labels: {{ color: '#8060a0', font: {{ size: 9 }}, boxWidth: 10, padding: 6 }} }} }}
  }}
}});

new Chart(document.getElementById('rankChart'), {{
  type: 'bar',
  data: {{
    labels: CHART_DATA.ranks.map(d => d.rank),
    datasets: [{{ data: CHART_DATA.ranks.map(d => d.count),
      backgroundColor: '#c9a0dcaa', borderColor: '#c9a0dc',
      borderWidth: 1, borderRadius: 4, borderSkipped: false }}]
  }},
  options: {{ responsive: true,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ grid: {{ display: false }}, ticks: {{ color: '#c9a0dc' }} }},
      y: {{ grid: {{ color: 'rgba(201,160,220,0.06)' }}, ticks: {{ color: '#8060a0' }} }}
    }}
  }}
}});

new Chart(document.getElementById('ingChart'), {{
  type: 'bar',
  data: {{
    labels: CHART_DATA.ingredients.map(d => d.name),
    datasets: [{{ data: CHART_DATA.ingredients.map(d => d.count),
      backgroundColor: '#e8d0f5aa', borderColor: '#e8d0f5',
      borderWidth: 1, borderRadius: 4, borderSkipped: false }}]
  }},
  options: {{ indexAxis: 'y', responsive: true,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ grid: {{ color: 'rgba(201,160,220,0.06)' }}, ticks: {{ color: '#8060a0' }} }},
      y: {{ grid: {{ display: false }}, ticks: {{ color: '#e8d0f5', font: {{ size: 9 }} }} }}
    }}
  }}
}});

new Chart(document.getElementById('benefitChart'), {{
  type: 'bar',
  data: {{
    labels: Object.keys(L2_DATA.benefit_counts),
    datasets: [{{ data: Object.values(L2_DATA.benefit_counts),
      backgroundColor: ['#c9a0dcaa','#e8d0f5aa','#f4c2ceaa','#9b7fd4aa','#d4a0c8aa','#b888d0aa','#e8b8d8aa'],
      borderRadius: 4, borderSkipped: false }}]
  }},
  options: {{ responsive: true,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ grid: {{ display: false }}, ticks: {{ color: '#c9a0dc', font: {{ size: 9 }} }} }},
      y: {{ grid: {{ color: 'rgba(201,160,220,0.06)' }}, ticks: {{ color: '#8060a0' }} }}
    }}
  }}
}});

// Top Products
document.getElementById('topProductsList').innerHTML = L2_DATA.top_products.slice(0,6).map((p,i) => `
  <div style="display:flex;align-items:center;gap:8px;padding:7px 0;border-bottom:1px solid rgba(201,160,220,0.06);">
    <div style="font-family:'Space Mono',monospace;font-size:11px;color:#c9a0dc;min-width:22px;">#${{i+1}}</div>
    <div style="flex:1;">
      <div style="font-size:11px;color:#f0e0ff;font-weight:600;">${{p.Name.length>35?p.Name.slice(0,35)+'...':p.Name}}</div>
      <div style="font-size:10px;color:#8060a0;margin-top:2px;">${{p.Brand}} · $${{p.Price}}</div>
      <div style="display:flex;flex-wrap:wrap;gap:3px;margin-top:3px;">
        ${{p.key_ingredients.map(ki=>`<span style="padding:1px 6px;background:rgba(201,160,220,0.1);border:1px solid rgba(201,160,220,0.2);border-radius:3px;font-size:9px;color:#c9a0dc;">${{ki.ingredient}}</span>`).join('')}}
      </div>
    </div>
    <div style="text-align:center;"><div style="font-family:'Space Mono',monospace;font-size:16px;font-weight:700;color:#c9a0dc;">${{p.score}}</div><div style="font-size:9px;color:#8060a0;">score</div></div>
  </div>`).join('');

// Heatmap
const brands = L2_DATA.brands;
const hdata = L2_DATA.heatmap;
const cs = Math.floor(260/brands.length);
let hmHTML = `<div style="display:grid;grid-template-columns:80px ${{brands.map(()=>cs+'px').join(' ')}};gap:2px;font-family:'Space Mono',monospace;font-size:8px;">`;
hmHTML += `<div></div>${{brands.map(b=>`<div style="color:#8060a0;text-align:center;height:36px;display:flex;align-items:flex-end;justify-content:center;padding-bottom:2px;">${{b.slice(0,7)}}</div>`).join('')}}`;
brands.forEach((b,i)=>{{
  hmHTML+=`<div style="color:#8060a0;display:flex;align-items:center;font-size:8px;">${{b.slice(0,9)}}</div>`;
  hdata[i].forEach((val,j)=>{{
    const intensity=val/100;
    const r=Math.round(201+54*intensity);
    const g=Math.round(160-94*intensity);
    const bl=Math.round(220-56*intensity);
    hmHTML+=`<div title="${{brands[i]}} vs ${{brands[j]}}: ${{val}}%" style="width:${{cs}}px;height:${{cs}}px;background:rgb(${{r}},${{g}},${{bl}});border-radius:2px;opacity:0.85;display:flex;align-items:center;justify-content:center;font-size:7px;color:#130d1a;font-weight:700;">${{val>10?Math.round(val)+'%':''}}</div>`;
  }});
}});
hmHTML+='</div>';
document.getElementById('heatmapContainer').innerHTML=hmHTML;

// Ingredient search
document.getElementById('ingSearch').addEventListener('input',function(){{
  const q=this.value.toLowerCase().trim();
  const res=document.getElementById('ingResults');
  if(!q){{res.innerHTML='';return;}}
  const matches=PRODUCTS.filter(p=>p.Ingredients.toLowerCase().includes(q)).slice(0,5);
  res.innerHTML=matches.length===0
    ?`<div style="color:#8060a0;font-size:11px;font-family:'Space Mono',monospace;">No products found</div>`
    :matches.map(p=>`<div style="padding:7px;background:#130d1a;border:1px solid rgba(201,160,220,0.1);border-radius:7px;">
      <div style="font-size:11px;color:#f0e0ff;font-weight:600;">${{p.Name.length>40?p.Name.slice(0,40)+'...':p.Name}}</div>
      <div style="font-size:10px;color:#8060a0;font-family:'Space Mono',monospace;">${{p.Brand}} · $${{p.Price}} · ${{p.Rank}} stars</div>
    </div>`).join('');
}});

// Recommendation engine
function toggleConcern(el,concern){{
  if(selectedConcerns.has(concern)){{selectedConcerns.delete(concern);el.style.background='transparent';el.style.color='#8060a0';el.style.borderColor='rgba(244,194,206,0.3)';}}
  else{{selectedConcerns.add(concern);el.style.background='#c9a0dc';el.style.color='#130d1a';el.style.borderColor='#c9a0dc';}}
}}
function updateBudget(el){{
  maxBudget=parseInt(el.value);
  const pct=(maxBudget-10)/290*100;
  el.style.background=`linear-gradient(to right,#c9a0dc 0%,#c9a0dc ${{pct}}%,#261530 ${{pct}}%)`;
  document.getElementById('budgetLabel').textContent='$'+maxBudget;
}}
function getRecommendations(){{
  const res=document.getElementById('recResults');
  if(selectedConcerns.size===0){{res.innerHTML='<div style="color:#f4c2ce;font-size:11px;text-align:center;padding:10px;">Please select a skin concern first!</div>';return;}}
  const targetIngs=new Set();
  selectedConcerns.forEach(c=>{{REC_DATA.concern_ingredients[c].forEach(i=>targetIngs.add(i));}});
  const scored=PRODUCTS.filter(p=>p.Price<=maxBudget).map(p=>{{
    const ingLower=p.Ingredients.toLowerCase();
    let score=0;targetIngs.forEach(ing=>{{if(ingLower.includes(ing))score++;}});
    return{{...p,recScore:score}};
  }}).filter(p=>p.recScore>0).sort((a,b)=>b.recScore-a.recScore||b.Rank-a.Rank).slice(0,5);
  if(scored.length===0){{res.innerHTML='<div style="color:#8060a0;font-size:11px;text-align:center;padding:10px;">No products found. Try increasing budget!</div>';return;}}
  res.innerHTML=scored.map((p,i)=>`
    <div style="padding:9px;background:#130d1a;border:1px solid rgba(201,160,220,0.15);border-radius:8px;display:flex;align-items:center;gap:8px;">
      <div style="font-family:'Space Mono',monospace;font-size:13px;color:#c9a0dc;font-weight:700;min-width:22px;">#${{i+1}}</div>
      <div style="flex:1;">
        <div style="font-size:11px;color:#f0e0ff;font-weight:600;">${{p.Name.length>38?p.Name.slice(0,38)+'...':p.Name}}</div>
        <div style="font-size:10px;color:#8060a0;font-family:'Space Mono',monospace;margin-top:2px;">${{p.Brand}} · $${{p.Price}} · ${{p.Rank}} stars</div>
      </div>
      <div style="text-align:center;">
        <div style="font-family:'Space Mono',monospace;font-size:15px;font-weight:700;color:#c9a0dc;">${{p.recScore}}</div>
        <div style="font-size:9px;color:#8060a0;">matches</div>
      </div>
    </div>`).join('');
}}

// Dupes finder
function searchProducts(query){{
  const dd=document.getElementById('productDropdown');
  if(!query||query.length<2){{dd.innerHTML='';return;}}
  const matches=PRODUCTS.map((p,i)=>{{return{{...p,idx:i}}}}).filter(p=>p.Name.toLowerCase().includes(query.toLowerCase())).slice(0,5);
  dd.innerHTML=matches.map(p=>`
    <div onclick="selectProductForDupe(${{p.idx}},'${{p.Name.replace(/'/g,"\\'")}}',this)"
      style="padding:7px 9px;background:#130d1a;border:1px solid rgba(201,160,220,0.1);border-radius:6px;cursor:pointer;font-size:11px;color:#f0e0ff;"
      onmouseover="this.style.borderColor='#c9a0dc'" onmouseout="this.style.borderColor='rgba(201,160,220,0.1)'">
      <div style="font-weight:600;">${{p.Name.length>42?p.Name.slice(0,42)+'...':p.Name}}</div>
      <div style="font-size:10px;color:#8060a0;font-family:'Space Mono',monospace;">${{p.Brand}} · $${{p.Price}}</div>
    </div>`).join('');
}}
function selectProductForDupe(idx,name){{
  document.getElementById('dupeSearch').value=name;
  document.getElementById('productDropdown').innerHTML='';
  const recs=REC_DATA.recommendations[idx];
  document.getElementById('dupeResults').innerHTML=`
    <div style="font-family:'Space Mono',monospace;font-size:10px;color:#c9a0dc;letter-spacing:1px;margin-bottom:8px;">SIMILAR PRODUCTS</div>
    ${{recs.map((r,i)=>`
      <div style="padding:8px;background:#130d1a;border:1px solid rgba(201,160,220,0.1);border-radius:7px;display:flex;align-items:center;gap:8px;margin-bottom:5px;">
        <div style="flex:1;">
          <div style="font-size:11px;color:#f0e0ff;font-weight:600;">${{r.Name.length>38?r.Name.slice(0,38)+'...':r.Name}}</div>
          <div style="font-size:10px;color:#8060a0;font-family:'Space Mono',monospace;margin-top:2px;">${{r.Brand}} · $${{r.Price}}</div>
        </div>
        <div style="text-align:center;">
          <div style="font-family:'Space Mono',monospace;font-size:13px;font-weight:700;color:${{r.similarity>50?'#c9a0dc':'#e8d0f5'}}">${{r.similarity}}%</div>
          <div style="font-size:9px;color:#8060a0;">match</div>
        </div>
      </div>`).join('')}}`;
}}
</script>
'''

html = html.replace('</body>', all_sections + '</body>')

with open('cosmetic_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(html)

print("✅ All sections added back!")
print("Refresh your browser and scroll down!")

✅ All sections added back!
Refresh your browser and scroll down!


In [73]:
with open('cosmetic_dashboard.html', 'r', encoding='utf-8') as f:
    html = f.read()

# Replace all emojis with text alternatives
replacements = [
    ('✦', '*'),
    ('&#9906;', '&#128269;'),
    ('&#8594;', '->'),
    ('💜', ''),
    ('🌸', ''),
    ('💗', ''),
    ('✨', ''),
    ('🧬', ''),
    ('🔍', ''),
    ('⬇', ''),
    ('🔗', ''),
]

for old, new in replacements:
    html = html.replace(old, new)

# Remove any remaining emoji characters
import re
emoji_pattern = re.compile("["
    u"\U0001F600-\U0001F64F"
    u"\U0001F300-\U0001F5FF"
    u"\U0001F680-\U0001F9FF"
    u"\U00002702-\U000027B0"
    u"\U000024C2-\U0001F251"
    "]+", flags=re.UNICODE)

html = emoji_pattern.sub('', html)

# Save with explicit UTF-8 BOM to force correct encoding
with open('cosmetic_dashboard.html', 'w', encoding='utf-8-sig') as f:
    f.write(html)

print("✅ All emojis removed!")
print("Now upload to Google Cloud again!")

✅ All emojis removed!
Now upload to Google Cloud again!


In [74]:
with open('cosmetic_dashboard.html', 'r', encoding='utf-8') as f:
    content = f.read()

print("File size:", len(content), "characters")
print("Has charset meta tag:", '<meta charset' in content)
print("First 200 chars of file:")
print(repr(content[:200]))

File size: 376322 characters
Has charset meta tag: True
First 200 chars of file:
'\ufeff<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta charset="UTF-8">\n<meta name="viewport" content="width=device-width, initial-scale=1.0">\n<title>CosmeticLab - Lavender Dreams</title>\n<link href="https://'


In [75]:
# Read and remove the BOM
with open('cosmetic_dashboard.html', 'r', encoding='utf-8-sig') as f:
    content = f.read()

# Save WITHOUT BOM
with open('cosmetic_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(content)

print("✅ BOM removed!")
print("File starts with:", repr(content[:50]))

✅ BOM removed!
File starts with: '<!DOCTYPE html>\n<html lang="en">\n<head>\n<meta char'


In [76]:
with open('cosmetic_dashboard.html', 'r', encoding='utf-8') as f:
    content = f.read()

# Replace ALL problematic characters with plain HTML alternatives
fixes = [
    # Middle dots and special chars
    ('·', '&middot;'),
    ('✦', '+'),
    ('→', '&rarr;'),
    ('★', '*'),
    ('◈', '#'),
    ('❧', '~'),
    # Smart quotes or anything non-ASCII in text content
    ('\u00b7', '&middot;'),
    ('\u2022', '-'),
    ('\u2014', '--'),
    ('\u2013', '-'),
]

for old, new in fixes:
    content = content.replace(old, new)

# Make absolutely sure no BOM
content = content.lstrip('\ufeff')

# Verify it's clean
import re
non_ascii = re.findall(r'[^\x00-\x7F]', content[:5000])
print("Non-ASCII chars in first 5000:", set(non_ascii))

with open('cosmetic_dashboard.html', 'w', encoding='ascii', errors='xmlcharrefreplace') as f:
    f.write(content)

print("✅ Saved as pure ASCII!")


Non-ASCII chars in first 5000: set()
✅ Saved as pure ASCII!


In [77]:
import json
import re

# Get fresh clean data - encode special chars
data = moisturizers_dry[['Name','Brand','Price','Rank','Ingredients',
                          'Dry','Normal','Oily','Combination','Sensitive',
                          'X','Y']].to_dict('records')

for p in data:
    p['Price'] = float(p['Price'])
    p['Rank'] = float(p['Rank'])
    for k in ['Dry','Normal','Oily','Combination','Sensitive']:
        p[k] = int(p[k])
    # Clean special characters from text fields
    for k in ['Name','Brand','Ingredients']:
        p[k] = p[k].encode('ascii', 'ignore').decode('ascii')

# Save as JSON to check
data_json = json.dumps(data, ensure_ascii=True)

# Check for problems
non_ascii = [c for c in data_json if ord(c) > 127]
print("Non-ASCII chars remaining:", len(non_ascii))
print("Sample product name:", data[0]['Name'])
print("✅ Data cleaned!")

Non-ASCII chars remaining: 0
Sample product name: Crme de la Mer
✅ Data cleaned!


In [78]:
# Now rebuild the full dashboard with clean ASCII data
dashboard = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>CosmeticLab - Lavender Dreams</title>
<link href="https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Cormorant+Garamond:wght@400;600;700&family=Jost:wght@300;400;600;800&display=swap" rel="stylesheet">
<style>
:root{{--bg:#130d1a;--bg2:#1e1228;--bg3:#261530;--accent:#c9a0dc;--accent2:#e8d0f5;--accent3:#f4c2ce;--text:#f0e0ff;--muted:#8060a0;--border:rgba(201,160,220,0.2);--glow:rgba(201,160,220,0.35);}}
*{{margin:0;padding:0;box-sizing:border-box;}}
body{{background:var(--bg);color:var(--text);font-family:'Jost',sans-serif;min-height:100vh;overflow-x:hidden;}}
body::before{{content:'';position:fixed;inset:0;background-image:linear-gradient(rgba(201,160,220,0.04) 1px,transparent 1px),linear-gradient(90deg,rgba(201,160,220,0.04) 1px,transparent 1px);background-size:40px 40px;pointer-events:none;z-index:0;}}
header{{position:relative;z-index:10;padding:16px 28px;border-bottom:1px solid var(--border);display:flex;align-items:center;justify-content:space-between;background:rgba(19,13,26,0.95);backdrop-filter:blur(12px);}}
.logo{{display:flex;align-items:center;gap:12px;}}
.logo-icon{{width:36px;height:36px;background:var(--accent);border-radius:8px;display:flex;align-items:center;justify-content:center;font-size:16px;color:#130d1a;font-weight:900;}}
.logo-text{{font-family:'Cormorant Garamond',serif;font-size:22px;font-weight:700;color:var(--text);letter-spacing:1px;}}
.logo-text span{{color:var(--accent);}}
.header-stats{{display:flex;gap:28px;}}
.hstat{{text-align:center;}}
.hstat-num{{font-family:'Space Mono',monospace;font-size:20px;font-weight:700;color:var(--accent);}}
.hstat-label{{font-size:10px;color:var(--muted);letter-spacing:1px;text-transform:uppercase;}}
.layout{{position:relative;z-index:1;display:grid;grid-template-columns:260px 1fr 280px;height:calc(100vh - 69px);}}
.left-panel{{background:var(--bg2);border-right:1px solid var(--border);overflow-y:auto;padding:20px;}}
.panel-title{{font-size:10px;letter-spacing:2px;text-transform:uppercase;color:var(--accent);margin-bottom:16px;font-family:'Space Mono',monospace;}}
.search-wrap{{position:relative;margin-bottom:20px;}}
.search-wrap input{{width:100%;background:var(--bg3);border:1px solid var(--border);border-radius:8px;padding:10px 14px;color:var(--text);font-family:'Jost',sans-serif;font-size:13px;outline:none;transition:border-color 0.2s;}}
.search-wrap input:focus{{border-color:var(--accent);}}
.search-wrap input::placeholder{{color:var(--muted);}}
.filter-section{{margin-bottom:20px;}}
.filter-label{{font-size:10px;letter-spacing:1.5px;text-transform:uppercase;color:var(--muted);margin-bottom:8px;}}
.filter-chips{{display:flex;flex-wrap:wrap;gap:5px;}}
.chip{{padding:4px 12px;border-radius:20px;font-size:11px;border:1px solid var(--border);background:transparent;color:var(--muted);cursor:pointer;transition:all 0.2s;font-family:'Jost',sans-serif;}}
.chip:hover,.chip.active{{background:var(--accent);border-color:var(--accent);color:#130d1a;font-weight:600;}}
.range-slider{{-webkit-appearance:none;width:100%;height:3px;background:linear-gradient(to right,var(--accent) 0%,var(--accent) 100%,var(--bg3) 100%);border-radius:2px;outline:none;cursor:pointer;}}
.range-slider::-webkit-slider-thumb{{-webkit-appearance:none;width:14px;height:14px;border-radius:50%;background:var(--accent);cursor:pointer;}}
.price-labels{{display:flex;justify-content:space-between;font-size:10px;color:var(--muted);font-family:'Space Mono',monospace;margin-top:6px;}}
.brand-list{{display:flex;flex-direction:column;gap:3px;max-height:200px;overflow-y:auto;}}
.brand-item{{display:flex;align-items:center;gap:8px;padding:5px 6px;border-radius:6px;cursor:pointer;transition:background 0.15s;font-size:11px;}}
.brand-item:hover{{background:var(--bg3);}}
.brand-dot{{width:8px;height:8px;border-radius:50%;flex-shrink:0;}}
.brand-name{{color:var(--text);flex:1;}}
.brand-count{{font-family:'Space Mono',monospace;font-size:10px;color:var(--muted);}}
.center-panel{{display:flex;flex-direction:column;overflow:hidden;}}
.viz-toolbar{{padding:10px 16px;border-bottom:1px solid var(--border);display:flex;align-items:center;gap:8px;background:var(--bg2);flex-wrap:wrap;}}
.viz-label{{font-size:11px;color:var(--muted);margin-right:auto;font-family:'Space Mono',monospace;}}
.viz-btn{{padding:5px 12px;border-radius:6px;font-size:11px;border:1px solid var(--border);background:transparent;color:var(--muted);cursor:pointer;font-family:'Jost',sans-serif;transition:all 0.2s;}}
.viz-btn:hover,.viz-btn.active{{border-color:var(--accent);color:var(--accent);}}
canvas#scatterCanvas{{flex:1;display:block;cursor:crosshair;}}
.right-panel{{background:var(--bg2);border-left:1px solid var(--border);overflow-y:auto;display:flex;flex-direction:column;}}
.stats-row{{display:grid;grid-template-columns:1fr 1fr;gap:1px;border-bottom:1px solid var(--border);}}
.stat-card{{padding:14px;background:var(--bg2);}}
.stat-num{{font-family:'Space Mono',monospace;font-size:20px;font-weight:700;color:var(--accent);}}
.stat-num.teal{{color:var(--accent2);}}
.stat-num.pink{{color:var(--accent3);}}
.stat-desc{{font-size:10px;color:var(--muted);margin-top:2px;}}
.detail-area{{flex:1;padding:16px;}}
.no-selection{{height:200px;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:10px;color:var(--muted);text-align:center;}}
.product-card{{display:none;}}
.product-card.visible{{display:block;}}
.product-brand{{font-size:10px;letter-spacing:2px;text-transform:uppercase;color:var(--accent);font-family:'Space Mono',monospace;margin-bottom:4px;}}
.product-name{{font-size:15px;font-weight:700;line-height:1.3;color:var(--text);margin-bottom:10px;}}
.product-meta{{display:flex;gap:8px;margin-bottom:14px;}}
.meta-pill{{padding:3px 9px;border-radius:20px;font-size:11px;font-family:'Space Mono',monospace;}}
.meta-pill.price{{background:rgba(201,160,220,0.1);color:var(--accent);border:1px solid rgba(201,160,220,0.2);}}
.meta-pill.rank{{background:rgba(232,208,245,0.1);color:var(--accent2);border:1px solid rgba(232,208,245,0.2);}}
.section-title{{font-size:10px;letter-spacing:2px;text-transform:uppercase;color:var(--muted);margin-bottom:8px;}}
.ingredient-tags{{display:flex;flex-wrap:wrap;gap:4px;margin-bottom:14px;}}
.ing-tag{{padding:3px 7px;background:var(--bg3);border:1px solid var(--border);border-radius:4px;font-size:10px;color:var(--muted);font-family:'Space Mono',monospace;}}
.ing-tag.highlight{{background:rgba(201,160,220,0.08);border-color:rgba(201,160,220,0.3);color:var(--accent);}}
.similar-list{{display:flex;flex-direction:column;gap:5px;}}
.similar-item{{padding:8px;background:var(--bg3);border:1px solid var(--border);border-radius:8px;cursor:pointer;transition:all 0.2s;}}
.similar-item:hover{{border-color:var(--accent);}}
.similar-item-brand{{font-size:9px;color:var(--accent);letter-spacing:1px;text-transform:uppercase;}}
.similar-item-name{{font-size:11px;color:var(--text);margin-top:2px;}}
.similar-item-price{{font-size:10px;color:var(--muted);font-family:'Space Mono',monospace;margin-top:3px;}}
#tooltip{{position:fixed;background:var(--bg2);border:1px solid var(--accent);border-radius:8px;padding:8px 12px;pointer-events:none;z-index:1000;display:none;max-width:220px;box-shadow:0 0 20px var(--glow);}}
.tt-brand{{font-size:9px;color:var(--accent);letter-spacing:1.5px;text-transform:uppercase;font-family:'Space Mono',monospace;}}
.tt-name{{font-size:12px;color:var(--text);margin-top:2px;font-weight:600;}}
.tt-price{{font-size:11px;color:var(--accent2);margin-top:3px;font-family:'Space Mono',monospace;}}
::-webkit-scrollbar{{width:3px;}}
::-webkit-scrollbar-track{{background:var(--bg);}}
::-webkit-scrollbar-thumb{{background:var(--border);border-radius:2px;}}
.bottom-bar{{position:fixed;bottom:0;left:0;right:0;height:28px;background:var(--bg2);border-top:1px solid var(--border);display:flex;align-items:center;padding:0 20px;gap:20px;z-index:100;font-family:'Space Mono',monospace;font-size:10px;color:var(--muted);}}
.status-dot{{width:6px;height:6px;border-radius:50%;background:var(--accent);animation:pulse 2s infinite;}}
@keyframes pulse{{0%,100%{{opacity:1;}}50%{{opacity:0.4;}}}}
@media(max-width:768px){{
  .layout{{grid-template-columns:1fr;height:auto;}}
  .left-panel{{border-right:none;border-bottom:1px solid var(--border);max-height:280px;}}
  .right-panel{{border-left:none;border-top:1px solid var(--border);}}
  canvas#scatterCanvas{{height:380px;}}
  .header-stats{{gap:14px;}}
}}
</style>
</head>
<body>

<!-- Loading Screen -->
<div id="loadingScreen" style="position:fixed;inset:0;background:#130d1a;z-index:9999;display:flex;flex-direction:column;align-items:center;justify-content:center;">
  <div style="font-family:'Cormorant Garamond',serif;font-size:48px;font-weight:700;color:#c9a0dc;letter-spacing:4px;margin-bottom:8px;">CosmeticLab</div>
  <div style="font-family:'Space Mono',monospace;font-size:11px;color:#8060a0;letter-spacing:3px;margin-bottom:40px;">INGREDIENT INTELLIGENCE</div>
  <div style="display:flex;gap:8px;">
    <div style="width:8px;height:8px;border-radius:50%;background:#c9a0dc;animation:pulse 1.2s ease infinite;"></div>
    <div style="width:8px;height:8px;border-radius:50%;background:#e8d0f5;animation:pulse 1.2s ease infinite 0.2s;"></div>
    <div style="width:8px;height:8px;border-radius:50%;background:#f4c2ce;animation:pulse 1.2s ease infinite 0.4s;"></div>
  </div>
</div>

<!-- Sparkle Canvas -->
<canvas id="sparkleCanvas" style="position:fixed;inset:0;pointer-events:none;z-index:0;opacity:0.5;"></canvas>

<header>
  <div class="logo">
    <div class="logo-icon">CL</div>
    <div class="logo-text">Cosmetic<span>Lab</span></div>
  </div>
  <div class="header-stats">
    <div class="hstat"><div class="hstat-num" id="totalCount">190</div><div class="hstat-label">Products</div></div>
    <div class="hstat"><div class="hstat-num" id="brandCount">--</div><div class="hstat-label">Brands</div></div>
    <div class="hstat"><div class="hstat-num" id="visibleCount">190</div><div class="hstat-label">Visible</div></div>
    <div class="hstat"><div class="hstat-num" id="avgPrice">--</div><div class="hstat-label">Avg Price</div></div>
  </div>
</header>

<div class="layout">
  <div class="left-panel">
    <div class="panel-title">// Filters</div>
    <div class="search-wrap">
      <input type="text" id="searchInput" placeholder="Search products or brands..." oninput="applyFilters()">
    </div>
    <div class="filter-section">
      <div class="filter-label">Skin Type</div>
      <div class="filter-chips">
        <div class="chip active" onclick="toggleSkin(this,'all')">All</div>
        <div class="chip" onclick="toggleSkin(this,'Dry')">Dry</div>
        <div class="chip" onclick="toggleSkin(this,'Normal')">Normal</div>
        <div class="chip" onclick="toggleSkin(this,'Oily')">Oily</div>
        <div class="chip" onclick="toggleSkin(this,'Combination')">Combo</div>
        <div class="chip" onclick="toggleSkin(this,'Sensitive')">Sensitive</div>
      </div>
    </div>
    <div class="filter-section">
      <div class="filter-label">Price Range</div>
      <input type="range" class="range-slider" id="priceRange" min="0" max="500" value="500" step="1" oninput="updatePrice(this)">
      <div class="price-labels"><span>$0</span><span id="priceLabel">Any price</span></div>
    </div>
    <div class="filter-section">
      <div class="filter-label">Color By</div>
      <div class="filter-chips">
        <div class="chip active" onclick="setColorMode(this,'brand')">Brand</div>
        <div class="chip" onclick="setColorMode(this,'price')">Price</div>
        <div class="chip" onclick="setColorMode(this,'rank')">Rank</div>
      </div>
    </div>
    <div class="filter-section">
      <div class="filter-label">Brands</div>
      <div class="brand-list" id="brandList"></div>
    </div>
  </div>

  <div class="center-panel">
    <div class="viz-toolbar">
      <span class="viz-label">t-SNE - Ingredient Similarity Map</span>
      <button class="viz-btn active" onclick="setVizMode(this,'scatter')">Scatter</button>
      <button class="viz-btn" onclick="setVizMode(this,'density')">Density</button>
      <button class="viz-btn" onclick="resetZoom()">Reset View</button>
      <button class="viz-btn" onclick="downloadChart()">Download PNG</button>
      <button class="viz-btn" onclick="shareURL()" id="shareBtn">Share Link</button>
    </div>
    <canvas id="scatterCanvas"></canvas>
  </div>

  <div class="right-panel">
    <div class="stats-row">
      <div class="stat-card"><div class="stat-num" id="statAvgRank">--</div><div class="stat-desc">Avg Rank</div></div>
      <div class="stat-card"><div class="stat-num teal" id="statMinPrice">--</div><div class="stat-desc">Min Price</div></div>
      <div class="stat-card"><div class="stat-num pink" id="statMaxPrice">--</div><div class="stat-desc">Max Price</div></div>
      <div class="stat-card"><div class="stat-num" id="statIngCount">--</div><div class="stat-desc">Avg Ingredients</div></div>
    </div>
    <div class="detail-area">
      <div class="no-selection" id="noSelection">
        <div style="font-size:32px;opacity:0.3;">+</div>
        <div style="font-size:12px;color:var(--muted);text-align:center;line-height:1.6;">Click any dot on the map<br>to explore product details</div>
      </div>
      <div class="product-card" id="productCard">
        <div class="product-brand" id="pdBrand"></div>
        <div class="product-name" id="pdName"></div>
        <div class="product-meta">
          <div class="meta-pill price" id="pdPrice"></div>
          <div class="meta-pill rank" id="pdRank"></div>
        </div>
        <div class="section-title">// Ingredients</div>
        <div class="ingredient-tags" id="pdIngredients"></div>
        <div class="section-title">// Similar Products</div>
        <div class="similar-list" id="similarList"></div>
      </div>
    </div>
  </div>
</div>

<div id="tooltip">
  <div class="tt-brand" id="ttBrand"></div>
  <div class="tt-name" id="ttName"></div>
  <div class="tt-price" id="ttPrice"></div>
</div>

<div class="bottom-bar">
  <div class="status-dot"></div>
  <span style="color:var(--accent);">COSMETICLAB</span>
  <span>Sephora - 190 Moisturizers - Dry Skin</span>
  <span style="margin-left:auto">Visible: <span style="color:var(--accent);" id="footerVisible">190</span></span>
</div>

<script>
const PRODUCTS = {data_json};
const BRAND_COLORS = ['#c9a0dc','#e8d0f5','#f4c2ce','#9b7fd4','#d4a0c8','#b888d0','#e8b8d8','#a870c0','#f0c8e8','#c0a0e8','#d8b0f0','#e0c0d8','#b898c8','#f8d0e8','#c8b0e0','#e0a8d0','#d0b8e8','#b8a0d8','#e8c0e0','#c8a8d8','#d8c0f0','#b8b0d8','#e8d0e8'];
let state={{search:'',skinFilter:'all',maxPrice:500,colorMode:'brand',selectedBrands:new Set(),vizMode:'scatter',selectedIdx:-1,zoom:1,panX:0,panY:0,dragging:false,dragStart:null}};
let filteredIndices=[],brandColorMap={{}},canvas,ctx,hoveredIdx=-1;

window.addEventListener('load',()=>{{
  setTimeout(()=>{{
    const ls=document.getElementById('loadingScreen');
    ls.style.transition='opacity 0.8s';
    ls.style.opacity='0';
    setTimeout(()=>ls.style.display='none',800);
  }},2200);
  canvas=document.getElementById('scatterCanvas');
  ctx=canvas.getContext('2d');
  resizeCanvas();
  window.addEventListener('resize',resizeCanvas);
  canvas.addEventListener('mousemove',onMouseMove);
  canvas.addEventListener('click',onCanvasClick);
  canvas.addEventListener('wheel',onWheel,{{passive:false}});
  canvas.addEventListener('mousedown',e=>{{state.dragging=true;state.dragStart={{x:e.clientX-state.panX,y:e.clientY-state.panY}};}});
  canvas.addEventListener('mouseup',()=>state.dragging=false);
  canvas.addEventListener('mouseleave',()=>{{state.dragging=false;document.getElementById('tooltip').style.display='none';}});
  initSparkles();
  buildBrandList();
  updateHeaderStats();
  applyFilters();
}});

function initSparkles(){{
  const sc=document.getElementById('sparkleCanvas');
  const sctx=sc.getContext('2d');
  sc.width=window.innerWidth;sc.height=window.innerHeight;
  window.addEventListener('resize',()=>{{sc.width=window.innerWidth;sc.height=window.innerHeight;}});
  const colors=['#c9a0dc','#e8d0f5','#f4c2ce','#9b7fd4'];
  const particles=Array.from({{length:50}},()=>{{
    return{{x:Math.random()*sc.width,y:Math.random()*sc.height,r:Math.random()*1.5+0.3,dx:(Math.random()-0.5)*0.2,dy:(Math.random()-0.5)*0.2,op:Math.random(),dop:(Math.random()-0.5)*0.008,color:colors[Math.floor(Math.random()*4)]}};
  }});
  function animSpark(){{
    sctx.clearRect(0,0,sc.width,sc.height);
    particles.forEach(p=>{{
      p.x+=p.dx;p.y+=p.dy;p.op+=p.dop;
      if(p.op<=0||p.op>=1)p.dop*=-1;
      if(p.x<0||p.x>sc.width)p.dx*=-1;
      if(p.y<0||p.y>sc.height)p.dy*=-1;
      sctx.beginPath();sctx.arc(p.x,p.y,p.r,0,Math.PI*2);
      sctx.fillStyle=p.color;sctx.globalAlpha=p.op;sctx.fill();
    }});
    sctx.globalAlpha=1;
    requestAnimationFrame(animSpark);
  }}
  animSpark();
}}

function buildBrandList(){{
  const counts={{}};
  PRODUCTS.forEach(p=>{{counts[p.Brand]=(counts[p.Brand]||0)+1;}});
  const bl=Object.entries(counts).sort((a,b)=>b[1]-a[1]);
  bl.forEach(([b],i)=>{{brandColorMap[b]=BRAND_COLORS[i%BRAND_COLORS.length];}});
  document.getElementById('brandList').innerHTML=bl.map(([b,c],i)=>`
    <div class="brand-item" onclick="toggleBrand('${{b.replace(/'/g,"\\'")}}',this)">
      <div class="brand-dot" style="background:${{BRAND_COLORS[i%BRAND_COLORS.length]}}"></div>
      <span class="brand-name">${{b}}</span>
      <span class="brand-count">${{c}}</span>
    </div>`).join('');
  document.getElementById('brandCount').textContent=bl.length;
}}

function updateHeaderStats(){{
  const prices=PRODUCTS.map(p=>p.Price);
  document.getElementById('avgPrice').textContent='$'+Math.round(prices.reduce((a,b)=>a+b,0)/prices.length);
  document.getElementById('statAvgRank').textContent=(PRODUCTS.map(p=>p.Rank).reduce((a,b)=>a+b,0)/PRODUCTS.length).toFixed(1)+'*';
  document.getElementById('statMinPrice').textContent='$'+Math.min(...prices);
  document.getElementById('statMaxPrice').textContent='$'+Math.max(...prices);
  const ings=PRODUCTS.map(p=>p.Ingredients.split(',').length);
  document.getElementById('statIngCount').textContent=Math.round(ings.reduce((a,b)=>a+b,0)/ings.length);
}}

function applyFilters(){{
  state.search=document.getElementById('searchInput').value.toLowerCase();
  filteredIndices=PRODUCTS.map((_,i)=>i).filter(i=>{{
    const p=PRODUCTS[i];
    if(state.search&&!p.Name.toLowerCase().includes(state.search)&&!p.Brand.toLowerCase().includes(state.search))return false;
    if(state.skinFilter!=='all'&&!p[state.skinFilter])return false;
    if(p.Price>state.maxPrice)return false;
    if(state.selectedBrands.size>0&&!state.selectedBrands.has(p.Brand))return false;
    return true;
  }});
  document.getElementById('visibleCount').textContent=filteredIndices.length;
  document.getElementById('footerVisible').textContent=filteredIndices.length;
  drawCanvas();
}}

function toggleSkin(el,val){{
  document.querySelectorAll('.filter-chips .chip').forEach(c=>{{if(c.onclick&&c.onclick.toString().includes('toggleSkin'))c.classList.remove('active');}});
  el.classList.add('active');state.skinFilter=val;applyFilters();
}}
function updatePrice(el){{
  state.maxPrice=parseInt(el.value);
  const pct=state.maxPrice/500*100;
  el.style.background='linear-gradient(to right,var(--accent) 0%,var(--accent) '+pct+'%,var(--bg3) '+pct+'%)';
  document.getElementById('priceLabel').textContent=state.maxPrice>=500?'Any price':'Up to $'+state.maxPrice;
  applyFilters();
}}
function toggleBrand(brand,el){{
  if(!el)return;
  el.classList.toggle('selected');
  if(state.selectedBrands.has(brand))state.selectedBrands.delete(brand);
  else state.selectedBrands.add(brand);
  applyFilters();
}}
function setColorMode(el,mode){{
  document.querySelectorAll('.filter-chips .chip').forEach(c=>{{if(c.onclick&&c.onclick.toString().includes('setColorMode'))c.classList.remove('active');}});
  el.classList.add('active');state.colorMode=mode;drawCanvas();
}}
function setVizMode(el,mode){{
  document.querySelectorAll('.viz-btn').forEach(b=>b.classList.remove('active'));
  el.classList.add('active');state.vizMode=mode;drawCanvas();
}}
function resetZoom(){{state.zoom=1;state.panX=0;state.panY=0;drawCanvas();}}
function resizeCanvas(){{
  const p=canvas.parentElement;
  canvas.width=p.clientWidth;
  canvas.height=p.clientHeight-41;
  drawCanvas();
}}
function getColor(p){{
  if(state.colorMode==='brand')return brandColorMap[p.Brand]||'#c9a0dc';
  if(state.colorMode==='price'){{
    const prices=PRODUCTS.map(q=>q.Price);
    const t=(p.Price-Math.min(...prices))/(Math.max(...prices)-Math.min(...prices));
    return 'rgb('+Math.round(201+54*t)+','+Math.round(160-94*t)+','+Math.round(220-56*t)+')';
  }}
  if(state.colorMode==='rank'){{
    const t=(p.Rank-3)/2;
    return 'rgb('+Math.round(244*t+201*(1-t))+','+Math.round(194*(1-t)+160*t)+','+Math.round(206*(1-t)+220*t)+')';
  }}
  return '#c9a0dc';
}}
function worldToScreen(x,y){{
  const cx=canvas.width/2+state.panX,cy=canvas.height/2+state.panY;
  const sc=(Math.min(canvas.width,canvas.height)/900)*state.zoom;
  return{{x:cx+x*sc,y:cy+y*sc}};
}}
function drawCanvas(){{
  if(!ctx)return;
  ctx.clearRect(0,0,canvas.width,canvas.height);
  const fs=new Set(filteredIndices);
  PRODUCTS.forEach((p,i)=>{{
    if(fs.has(i))return;
    const{{x,y}}=worldToScreen(p.X,p.Y);
    ctx.beginPath();ctx.arc(x,y,3,0,Math.PI*2);
    ctx.fillStyle='rgba(150,100,180,0.15)';ctx.fill();
  }});
  if(state.vizMode==='density'){{
    filteredIndices.forEach(i=>{{
      const p=PRODUCTS[i],{{x,y}}=worldToScreen(p.X,p.Y),color=getColor(p);
      const g=ctx.createRadialGradient(x,y,0,x,y,28);
      g.addColorStop(0,color+'40');g.addColorStop(1,color+'00');
      ctx.beginPath();ctx.arc(x,y,28,0,Math.PI*2);ctx.fillStyle=g;ctx.fill();
    }});
  }}
  filteredIndices.forEach(i=>{{
    const p=PRODUCTS[i],{{x,y}}=worldToScreen(p.X,p.Y),color=getColor(p),isSel=state.selectedIdx===i,r=isSel?10:7;
    if(isSel){{
      const g=ctx.createRadialGradient(x,y,0,x,y,22);
      g.addColorStop(0,color+'60');g.addColorStop(1,color+'00');
      ctx.beginPath();ctx.arc(x,y,22,0,Math.PI*2);ctx.fillStyle=g;ctx.fill();
    }}
    ctx.beginPath();ctx.arc(x,y,r,0,Math.PI*2);
    ctx.fillStyle=isSel?color:color+'dd';ctx.fill();
    if(isSel){{ctx.beginPath();ctx.arc(x,y,r+3,0,Math.PI*2);ctx.strokeStyle=color;ctx.lineWidth=1.5;ctx.stroke();}}
  }});
}}
function onMouseMove(e){{
  if(state.dragging&&state.dragStart){{
    state.panX=e.clientX-state.dragStart.x;state.panY=e.clientY-state.dragStart.y;drawCanvas();return;
  }}
  const rect=canvas.getBoundingClientRect(),mx=e.clientX-rect.left,my=e.clientY-rect.top;
  let closest=-1,cd=18;
  filteredIndices.forEach(i=>{{
    const p=PRODUCTS[i],{{x,y}}=worldToScreen(p.X,p.Y),d=Math.hypot(mx-x,my-y);
    if(d<cd){{closest=i;cd=d;}}
  }});
  const tt=document.getElementById('tooltip');
  if(closest>=0){{
    canvas.style.cursor='pointer';
    const p=PRODUCTS[closest];
    document.getElementById('ttBrand').textContent=p.Brand;
    document.getElementById('ttName').textContent=p.Name;
    document.getElementById('ttPrice').textContent='$'+p.Price+' - '+p.Rank+' stars';
    tt.style.display='block';tt.style.left=(e.clientX+14)+'px';tt.style.top=(e.clientY-10)+'px';
    hoveredIdx=closest;
  }}else{{canvas.style.cursor='crosshair';tt.style.display='none';hoveredIdx=-1;}}
}}
function onCanvasClick(){{
  if(hoveredIdx>=0){{state.selectedIdx=hoveredIdx;showDetail(PRODUCTS[hoveredIdx],hoveredIdx);drawCanvas();}}
}}
function onWheel(e){{
  e.preventDefault();state.zoom=Math.max(0.3,Math.min(5,state.zoom*(e.deltaY>0?0.9:1.1)));drawCanvas();
}}
function showDetail(p,idx){{
  document.getElementById('noSelection').style.display='none';
  const card=document.getElementById('productCard');card.classList.add('visible');
  document.getElementById('pdBrand').textContent=p.Brand;
  document.getElementById('pdName').textContent=p.Name;
  document.getElementById('pdPrice').textContent='$'+p.Price;
  document.getElementById('pdRank').textContent=p.Rank+' stars';
  const ings=p.Ingredients.split(',').map(s=>s.trim()).filter(Boolean);
  const keys=['Water','Glycerin','Niacinamide','Hyaluronic Acid','Ceramide','Retinol','Vitamin C','Peptides','Squalane'];
  document.getElementById('pdIngredients').innerHTML=ings.slice(0,18).map(ing=>'<span class="ing-tag '+(keys.some(k=>ing.toLowerCase().includes(k.toLowerCase()))?'highlight':'')+'">' +ing+'</span>').join('');
  const dists=PRODUCTS.map((q,i)=>{{return{{i,d:Math.hypot(q.X-p.X,q.Y-p.Y)}}}}).filter(d=>d.i!==idx).sort((a,b)=>a.d-b.d).slice(0,4);
  document.getElementById('similarList').innerHTML=dists.map(({{i}})=>{{
    const q=PRODUCTS[i];
    return '<div class="similar-item" onclick="selectProduct('+i+')"><div class="similar-item-brand">'+q.Brand+'</div><div class="similar-item-name">'+q.Name+'</div><div class="similar-item-price">$'+q.Price+' - '+q.Rank+' stars</div></div>';
  }}).join('');
}}
function selectProduct(idx){{state.selectedIdx=idx;showDetail(PRODUCTS[idx],idx);drawCanvas();}}
function downloadChart(){{
  const link=document.createElement('a');
  link.download='cosmetic_map.png';
  link.href=canvas.toDataURL('image/png');
  link.click();
}}
function shareURL(){{
  const url='https://storage.googleapis.com/cosmetic-map-aishu/cosmetic_dashboard.html';
  navigator.clipboard.writeText(url).then(()=>{{
    const btn=document.getElementById('shareBtn');
    btn.textContent='Copied!';btn.style.color='#c9a0dc';
    setTimeout(()=>{{btn.textContent='Share Link';btn.style.color='';}},2000);
  }});
}}
</script>
</body>
</html>'''

with open('cosmetic_dashboard.html', 'w', encoding='ascii', errors='xmlcharrefreplace') as f:
    f.write(dashboard)

print("✅ Clean ASCII dashboard saved!")
print("Now upload to Google Cloud!")

✅ Clean ASCII dashboard saved!
Now upload to Google Cloud!
